In [2]:
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# SYNAPSE-KG v2: Improved Version with Better Component Designs
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
#
# IMPROVEMENTS:
# 1. DRIP: Enhanced with proper KG message passing and attention
# 2. MINT: Fixed contrastive learning with proper hard negative mining
# 3. NEXIS: Improved capsule routing with interest extraction from history
# 4. BRIDGE: Optimized bidirectional propagation
# 5. ORACLE: Better fusion with component-specific normalization
#
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

import os
import sys
import io
import math
import random
import warnings
import zipfile
import urllib.request
import time
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy import stats, sparse as sp
from scipy.sparse import csr_matrix, coo_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    """Configuration for SYNAPSE-KG v2 model."""
    # Paths
    OUTPUT_PATH: str = "/content/drive/MyDrive/SYNAPSE/SYNAPSE-KG-v2-Results"
    ML_ROOT: str = "./data/ml-1m"
    ML_URL: str = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

    # Data settings
    SEED: int = 42
    SAMPLE_PCT: float = 0.35
    MIN_USER_INTER: int = 10
    MIN_ITEM_INTER: int = 5
    N_NEG: int = 99
    TOP_KS: List[int] = field(default_factory=lambda: [5, 10, 20])

    # DRIP settings - IMPROVED
    DRIP_DIM: int = 64
    DRIP_LAYERS: int = 2
    DRIP_HEADS: int = 4
    DRIP_EPOCHS: int = 50  # More epochs
    DRIP_LR: float = 5e-4  # Lower LR

    # MINT settings - IMPROVED
    MINT_DIM: int = 64
    MINT_MOMENTUM: float = 0.995  # Higher momentum
    MINT_EPOCHS: int = 40  # More epochs
    MINT_LR: float = 5e-4
    MINT_QUEUE_SIZE: int = 2048
    MINT_TEMPERATURE: float = 0.1  # Explicit temperature

    # BRIDGE settings - IMPROVED
    BRIDGE_DIM: int = 64
    BRIDGE_STEPS: int = 3
    BRIDGE_EPOCHS: int = 40
    BRIDGE_LR: float = 1e-3

    # NEXIS settings - IMPROVED
    NEXIS_DIM: int = 64
    NEXIS_CAPSULES: int = 8
    NEXIS_ROUTING_ITERS: int = 3
    NEXIS_EPOCHS: int = 40  # More epochs
    NEXIS_LR: float = 5e-4

    # ORACLE settings - IMPROVED
    ORACLE_EPOCHS: int = 100
    ORACLE_LR: float = 1e-3
    ORACLE_TEMP: float = 0.5
    ORACLE_HIDDEN: int = 64

    # Baseline settings
    LGCN_DIM: int = 64
    LGCN_LAYERS: int = 3
    LGCN_EPOCHS: int = 30
    LGCN_LR: float = 1e-3

    MF_EPOCHS: int = 40
    MF_DIM: int = 64
    MF_LR: float = 1e-3


CFG = Config()

# Create directories
os.makedirs(CFG.OUTPUT_PATH, exist_ok=True)
os.makedirs(f"{CFG.OUTPUT_PATH}/figures", exist_ok=True)
os.makedirs(f"{CFG.OUTPUT_PATH}/tables", exist_ok=True)
os.makedirs(CFG.ML_ROOT, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CFG.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("═" * 110)
print("  SYNAPSE-KG v2: Improved Knowledge Graph Enhanced Recommendations")
print("═" * 110)
print(f"  Device: {DEVICE} | PyTorch: {torch.__version__}")
print(f"  Output: {CFG.OUTPUT_PATH}")
print("═" * 110)

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 1: DATA LOADING & PREPROCESSING (Same as before)
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 110)
print("PHASE 1: DATA LOADING")
print("─" * 110)

def download_movielens():
    ratings_path = os.path.join(CFG.ML_ROOT, "ratings.dat")
    if not os.path.exists(ratings_path):
        print("[Data] Downloading MovieLens-1M...")
        with urllib.request.urlopen(CFG.ML_URL, timeout=120) as response:
            data = response.read()
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            for name in zf.namelist():
                if name.endswith('.dat'):
                    with zf.open(name) as src:
                        dest_path = os.path.join(CFG.ML_ROOT, os.path.basename(name))
                        with open(dest_path, 'wb') as dst:
                            dst.write(src.read())
        print("[Data] Download complete.")

def load_movielens():
    download_movielens()
    ratings = pd.read_csv(f"{CFG.ML_ROOT}/ratings.dat", sep="::", names=["userId", "movieId", "rating", "timestamp"], engine="python", encoding="latin-1")
    movies = pd.read_csv(f"{CFG.ML_ROOT}/movies.dat", sep="::", names=["movieId", "title", "genres"], engine="python", encoding="latin-1")
    return ratings, movies

ratings_df, movies_df = load_movielens()
print(f"[Data] Loaded: {len(ratings_df):,} ratings, {len(movies_df):,} movies, {ratings_df['userId'].nunique():,} users")

print("\n" + "─" * 110)
print("PHASE 1.5: DATA PREPROCESSING")
print("─" * 110)

def preprocess_data(ratings_df, movies_df):
    df = ratings_df[ratings_df['rating'] >= 4][['userId', 'movieId', 'timestamp', 'rating']].copy()

    for _ in range(3):
        user_counts = df.groupby('userId').size()
        df = df[df['userId'].isin(user_counts[user_counts >= CFG.MIN_USER_INTER].index)]
        item_counts = df.groupby('movieId').size()
        df = df[df['movieId'].isin(item_counts[item_counts >= CFG.MIN_ITEM_INTER].index)]

    rng = np.random.default_rng(CFG.SEED)
    all_users = df['userId'].unique()
    sampled_users = rng.choice(all_users, int(len(all_users) * CFG.SAMPLE_PCT), replace=False)
    df = df[df['userId'].isin(sampled_users)]

    user_counts = df.groupby('userId').size()
    df = df[df['userId'].isin(user_counts[user_counts >= CFG.MIN_USER_INTER].index)]

    users = sorted(df['userId'].unique())
    items = sorted(df['movieId'].unique())
    user_to_idx = {u: i for i, u in enumerate(users)}
    item_to_idx = {m: i for i, m in enumerate(items)}
    n_users, n_items = len(users), len(items)

    df['user'] = df['userId'].map(user_to_idx)
    df['item'] = df['movieId'].map(item_to_idx)
    df = df.sort_values(['user', 'timestamp']).reset_index(drop=True)

    user_sequences = {}
    for u in range(n_users):
        user_df = df[df['user'] == u].sort_values('timestamp')
        user_sequences[u] = user_df['item'].tolist()

    train_records, test_records = [], []
    for u in range(n_users):
        user_df = df[df['user'] == u].sort_values('timestamp')
        if len(user_df) >= 2:
            test_records.append(user_df.iloc[-1])
            train_records.extend(user_df.iloc[:-1].to_dict('records'))

    train_df = pd.DataFrame(train_records)
    test_df = pd.DataFrame(test_records)

    movies_subset = movies_df[movies_df['movieId'].isin(item_to_idx)].copy()
    movies_subset['item'] = movies_subset['movieId'].map(item_to_idx)

    train_set = set(zip(train_df['user'], train_df['item']))

    return {
        'train_df': train_df, 'test_df': test_df, 'movies': movies_subset,
        'n_users': n_users, 'n_items': n_items, 'user_to_idx': user_to_idx,
        'item_to_idx': item_to_idx, 'user_sequences': user_sequences, 'train_set': train_set
    }

data = preprocess_data(ratings_df, movies_df)
train_df, test_df, movies_subset = data['train_df'], data['test_df'], data['movies']
n_users, n_items = data['n_users'], data['n_items']
user_sequences, train_set = data['user_sequences'], data['train_set']

print(f"  Users: {n_users:,}, Items: {n_items:,}")
print(f"  Train: {len(train_df):,}, Test: {len(test_df):,}")
print(f"  Sparsity: {(1 - len(train_df) / (n_users * n_items)) * 100:.2f}%")

# Save Table 1
table1 = pd.DataFrame({
    'Metric': ['Users', 'Items', 'Train Interactions', 'Test Interactions', 'Sparsity (%)', 'Avg Items/User'],
    'Value': [n_users, n_items, len(train_df), len(test_df), f"{(1 - len(train_df) / (n_users * n_items)) * 100:.2f}", f"{len(train_df) / n_users:.1f}"]
})
table1.to_csv(f"{CFG.OUTPUT_PATH}/tables/Table1_Dataset_Statistics.csv", index=False)
print("[Saved] Table 1: Dataset Statistics")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 2: KNOWLEDGE GRAPH CONSTRUCTION
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 110)
print("PHASE 2: KNOWLEDGE GRAPH CONSTRUCTION")
print("─" * 110)

def build_knowledge_graph(train_df, movies_subset, n_users, n_items, user_sequences):
    triples = []
    rel_counts = defaultdict(int)

    USER_OFFSET, ITEM_OFFSET = 0, n_users

    all_genres = set()
    for genres in movies_subset['genres']:
        all_genres.update(genres.split('|'))
    genre_to_idx = {g: i for i, g in enumerate(sorted(all_genres))}
    n_genres = len(genre_to_idx)
    GENRE_OFFSET = ITEM_OFFSET + n_items

    for _, row in train_df.iterrows():
        u, i = int(row['user']), int(row['item'])
        triples.append((USER_OFFSET + u, 0, ITEM_OFFSET + i))
        rel_counts['INTERACTED'] += 1

    for _, row in movies_subset.iterrows():
        item_idx = int(row['item'])
        for genre in row['genres'].split('|'):
            if genre in genre_to_idx:
                triples.append((ITEM_OFFSET + item_idx, 1, GENRE_OFFSET + genre_to_idx[genre]))
                rel_counts['HAS_GENRE'] += 1

    co_watch_count = 0
    for u, items in user_sequences.items():
        if len(items) >= 2:
            for i in range(len(items)):
                for j in range(i + 1, min(i + 6, len(items))):
                    if co_watch_count < 25000:
                        triples.append((ITEM_OFFSET + items[i], 2, ITEM_OFFSET + items[j]))
                        co_watch_count += 1
    rel_counts['CO_WATCHED'] = co_watch_count

    item_genres = {int(row['item']): set(row['genres'].split('|')) for _, row in movies_subset.iterrows()}
    genre_sim_count = 0
    items_list = list(item_genres.keys())
    rng = np.random.default_rng(CFG.SEED)
    rng.shuffle(items_list)
    for i, item1 in enumerate(items_list[:500]):
        for item2 in items_list[i+1:i+50]:
            if item1 in item_genres and item2 in item_genres:
                if len(item_genres[item1] & item_genres[item2]) >= 2 and genre_sim_count < 10000:
                    triples.append((ITEM_OFFSET + item1, 3, ITEM_OFFSET + item2))
                    genre_sim_count += 1
    rel_counts['GENRE_SIMILAR'] = genre_sim_count

    temporal_count = 0
    for u, items in user_sequences.items():
        for i in range(len(items) - 1):
            triples.append((ITEM_OFFSET + items[i], 4, ITEM_OFFSET + items[i + 1]))
            temporal_count += 1
    rel_counts['TEMPORAL_NEXT'] = temporal_count

    return {
        'triples': triples, 'n_entities': GENRE_OFFSET + n_genres, 'n_relations': 5,
        'rel_counts': dict(rel_counts), 'USER_OFFSET': USER_OFFSET, 'ITEM_OFFSET': ITEM_OFFSET,
        'GENRE_OFFSET': GENRE_OFFSET, 'n_genres': n_genres, 'genre_to_idx': genre_to_idx
    }

kg = build_knowledge_graph(train_df, movies_subset, n_users, n_items, user_sequences)
print(f"  Entities: {kg['n_entities']:,}, Triples: {len(kg['triples']):,}")
print(f"  Relations: {kg['rel_counts']}")

table2 = pd.DataFrame([{'Relation': rel, 'Count': count} for rel, count in kg['rel_counts'].items()])
table2.to_csv(f"{CFG.OUTPUT_PATH}/tables/Table2_KG_Statistics.csv", index=False)
print("[Saved] Table 2: KG Statistics")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 3: FEATURE ENGINEERING & GRAPH STRUCTURES
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 110)
print("PHASE 3: FEATURE ENGINEERING")
print("─" * 110)

def build_features(train_df, movies_subset, n_users, n_items, user_sequences):
    print("[Features] Building content embeddings...")
    item_texts = []
    for i in range(n_items):
        row = movies_subset[movies_subset['item'] == i]
        text = row.iloc[0]['genres'].replace('|', ' ') + ' ' + row.iloc[0]['title'] if len(row) > 0 else 'unknown'
        item_texts.append(text)

    tfidf = TfidfVectorizer(max_features=500, stop_words='english')
    item_tfidf = tfidf.fit_transform(item_texts)
    svd = TruncatedSVD(n_components=64, random_state=CFG.SEED)
    item_content_emb = normalize(svd.fit_transform(item_tfidf))

    user_content_emb = np.zeros((n_users, 64))
    for u in range(n_users):
        items = user_sequences.get(u, [])
        if items:
            user_content_emb[u] = item_content_emb[items].mean(axis=0)
    user_content_emb = normalize(user_content_emb)

    print(f"  Content embeddings: users={user_content_emb.shape}, items={item_content_emb.shape}")

    print("[Features] Building popularity features...")
    item_pop = train_df.groupby('item').size().reindex(range(n_items), fill_value=0).values
    item_pop_norm = (item_pop - item_pop.min()) / (item_pop.max() - item_pop.min() + 1e-8)

    return {'user_content': user_content_emb, 'item_content': item_content_emb, 'item_popularity': item_pop_norm}

features = build_features(train_df, movies_subset, n_users, n_items, user_sequences)

print("\n" + "─" * 110)
print("PHASE 3.5: BUILDING GRAPH STRUCTURES")
print("─" * 110)

def build_graph_structures(train_df, n_users, n_items):
    rows = train_df['user'].values
    cols = train_df['item'].values + n_users
    data = np.ones(len(train_df))
    n_nodes = n_users + n_items
    adj = csr_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes))
    adj = adj + adj.T
    degrees = np.array(adj.sum(axis=1)).flatten()
    d_inv_sqrt = np.power(degrees + 1e-8, -0.5)
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    norm_adj = d_mat_inv_sqrt @ adj @ d_mat_inv_sqrt
    print(f"  Adjacency matrix: {adj.nnz:,} edges")
    return {'adj': adj, 'norm_adj': norm_adj, 'n_nodes': n_nodes}

graph = build_graph_structures(train_df, n_users, n_items)

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 4: EXTERNAL BASELINES
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("PHASE 4: EXTERNAL BASELINES")
print("═" * 110)

# ItemKNN
print("\n[Baseline 1] ItemKNN...")
class ItemKNN:
    def __init__(self, k=100):
        self.k = k
    def fit(self, train_df, n_users, n_items):
        self.n_users, self.n_items = n_users, n_items
        rows, cols = train_df['item'].values, train_df['user'].values
        item_user = csr_matrix((np.ones(len(train_df)), (rows, cols)), shape=(n_items, n_users))
        self.sim = normalize(item_user, axis=1) @ normalize(item_user, axis=1).T
        self.user_items = defaultdict(set)
        for _, row in train_df.iterrows():
            self.user_items[int(row['user'])].add(int(row['item']))
        return self
    def predict(self, user, items):
        history = list(self.user_items[user])
        if not history:
            return np.zeros(len(items))
        scores = []
        for item in items:
            sim_scores = self.sim[item, history].toarray().flatten()
            top_k = min(self.k, len(sim_scores))
            scores.append(np.partition(sim_scores, -top_k)[-top_k:].sum())
        return np.array(scores)

itemknn = ItemKNN(k=100).fit(train_df, n_users, n_items)
print("  ItemKNN ready.")

# UserKNN
print("[Baseline 2] UserKNN...")
class UserKNN:
    def __init__(self, k=50):
        self.k = k
    def fit(self, train_df, n_users, n_items):
        self.n_users, self.n_items = n_users, n_items
        rows, cols = train_df['user'].values, train_df['item'].values
        self.user_item = csr_matrix((np.ones(len(train_df)), (rows, cols)), shape=(n_users, n_items))
        self.sim = normalize(self.user_item, axis=1) @ normalize(self.user_item, axis=1).T
        return self
    def predict(self, user, items):
        sim_users = self.sim[user].toarray().flatten()
        top_k_idx = np.argsort(sim_users)[-self.k-1:-1]
        top_k_sim = sim_users[top_k_idx]
        scores = []
        for item in items:
            item_ratings = self.user_item[top_k_idx, item].toarray().flatten()
            scores.append((top_k_sim * item_ratings).sum() / (np.abs(top_k_sim).sum() + 1e-8))
        return np.array(scores)

userknn = UserKNN(k=50).fit(train_df, n_users, n_items)
print("  UserKNN ready.")

# MF-BPR
print("[Baseline 3] MF-BPR...")
class MFBPR(nn.Module):
    def __init__(self, n_users, n_items, dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
    def forward(self, users, pos_items, neg_items):
        u, pos, neg = self.user_emb(users), self.item_emb(pos_items), self.item_emb(neg_items)
        pos_scores = (u * pos).sum(dim=1)
        neg_scores = (u * neg).sum(dim=1)
        loss = -F.logsigmoid(pos_scores - neg_scores).mean()
        reg = 0.001 * (u.norm(2).pow(2) + pos.norm(2).pow(2) + neg.norm(2).pow(2)) / len(users)
        return loss + reg
    def get_embeddings(self):
        return self.user_emb.weight.detach().cpu().numpy(), self.item_emb.weight.detach().cpu().numpy()

def train_mf_bpr(n_users, n_items, train_df, train_set, epochs=40, dim=64, lr=1e-3):
    model = MFBPR(n_users, n_items, dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_users, train_items = train_df['user'].values, train_df['item'].values
    for epoch in range(epochs):
        model.train()
        idx = np.random.permutation(len(train_users))
        total_loss, batch_size = 0, 1024
        for i in range(0, len(idx), batch_size):
            batch_idx = idx[i:i+batch_size]
            users = torch.LongTensor(train_users[batch_idx]).to(DEVICE)
            pos_items = torch.LongTensor(train_items[batch_idx]).to(DEVICE)
            neg_items = []
            for u in train_users[batch_idx]:
                neg = np.random.randint(0, n_items)
                while (u, neg) in train_set:
                    neg = np.random.randint(0, n_items)
                neg_items.append(neg)
            neg_items = torch.LongTensor(neg_items).to(DEVICE)
            optimizer.zero_grad()
            loss = model(users, pos_items, neg_items)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={total_loss/(len(idx)//batch_size):.4f}")
    return model

mf_model = train_mf_bpr(n_users, n_items, train_df, train_set, epochs=CFG.MF_EPOCHS)
mf_user_emb, mf_item_emb = mf_model.get_embeddings()
print("  MF-BPR ready.")

# LightGCN
print("[Baseline 4] LightGCN...")
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, dim=64, n_layers=3):
        super().__init__()
        self.n_users, self.n_items, self.n_layers = n_users, n_items, n_layers
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
    def forward(self, norm_adj):
        all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        embs = [all_emb]
        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(norm_adj, all_emb)
            embs.append(all_emb)
        embs = torch.stack(embs, dim=1).mean(dim=1)
        return embs[:self.n_users], embs[self.n_users:]
    def bpr_loss(self, users, pos_items, neg_items, user_emb, item_emb):
        u, pos, neg = user_emb[users], item_emb[pos_items], item_emb[neg_items]
        loss = -F.logsigmoid((u * pos).sum(dim=1) - (u * neg).sum(dim=1)).mean()
        reg = 0.001 * (self.user_emb.weight[users].norm(2).pow(2) + self.item_emb.weight[pos_items].norm(2).pow(2) + self.item_emb.weight[neg_items].norm(2).pow(2)) / len(users)
        return loss + reg

def train_lightgcn(n_users, n_items, train_df, train_set, norm_adj, epochs=30, dim=64):
    coo = norm_adj.tocoo()
    norm_adj_t = torch.sparse_coo_tensor(torch.LongTensor(np.vstack([coo.row, coo.col])), torch.FloatTensor(coo.data), torch.Size(coo.shape)).to(DEVICE)
    model = LightGCN(n_users, n_items, dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LGCN_LR)
    train_users, train_items = train_df['user'].values, train_df['item'].values
    for epoch in range(epochs):
        model.train()
        user_emb, item_emb = model(norm_adj_t)
        idx = np.random.permutation(len(train_users))
        total_loss, batch_size = 0, 1024
        for i in range(0, len(idx), batch_size):
            batch_idx = idx[i:i+batch_size]
            users = torch.LongTensor(train_users[batch_idx]).to(DEVICE)
            pos_items = torch.LongTensor(train_items[batch_idx]).to(DEVICE)
            neg_items = []
            for u in train_users[batch_idx]:
                neg = np.random.randint(0, n_items)
                while (u, neg) in train_set:
                    neg = np.random.randint(0, n_items)
                neg_items.append(neg)
            neg_items = torch.LongTensor(neg_items).to(DEVICE)
            optimizer.zero_grad()
            loss = model.bpr_loss(users, pos_items, neg_items, user_emb, item_emb)
            loss.backward(retain_graph=True)
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={total_loss/(len(idx)//batch_size):.4f}")
    model.eval()
    with torch.no_grad():
        user_emb, item_emb = model(norm_adj_t)
    return model, user_emb.cpu().numpy(), item_emb.cpu().numpy()

lgcn_model, lgcn_user_emb, lgcn_item_emb = train_lightgcn(n_users, n_items, train_df, train_set, graph['norm_adj'], epochs=CFG.LGCN_EPOCHS)
print("  LightGCN ready.")

# Popularity
print("[Baseline 5] Popularity...")
item_popularity = train_df.groupby('item').size().reindex(range(n_items), fill_value=0).values
print("  Popularity ready.")

print("[Baseline 6] Content-Based...")
print("  Content-Based ready.")

training_history = {}
print("\nAll baselines ready. Proceeding to SYNAPSE-KG v2 components...")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 5: IMPROVED SYNAPSE-KG v2 COMPONENTS
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("PHASE 5: SYNAPSE-KG v2 IMPROVED COMPONENTS")
print("═" * 110)

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# COMPONENT 1: DRIP v2 (Improved Dynamic Relational Intent Propagation)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

print("\n" + "─" * 110)
print("COMPONENT 1: DRIP v2 (Improved Dynamic Relational Intent Propagation)")
print("  Improvement: Direct user-item scoring with attention over history")
print("─" * 110)

class DRIPv2(nn.Module):
    """Improved DRIP with better attention mechanism."""
    def __init__(self, n_users, n_items, dim=64, n_heads=4):
        super().__init__()
        self.n_users, self.n_items, self.dim = n_users, n_items, dim

        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)

        # Attention over user history
        self.attention = nn.MultiheadAttention(dim, n_heads, dropout=0.1, batch_first=True)
        self.query_proj = nn.Linear(dim, dim)

        # Intent-aware scoring
        self.intent_mlp = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(dim, dim)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.xavier_uniform_(m.weight)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, users, items, user_histories, history_mask):
        user_e = self.user_emb(users)
        item_e = self.item_emb(items)

        # Get history embeddings
        hist_emb = self.item_emb(user_histories)  # [B, L, D]

        # Query is target item
        query = self.query_proj(item_e).unsqueeze(1)  # [B, 1, D]

        # Attention over history
        attn_out, _ = self.attention(query, hist_emb, hist_emb, key_padding_mask=~history_mask)
        attn_out = attn_out.squeeze(1)  # [B, D]

        # Combine user, item, and attended history
        combined = torch.cat([user_e + attn_out, item_e], dim=-1)
        intent_repr = self.intent_mlp(combined)

        # Final scoring
        scores = (intent_repr * item_e).sum(dim=-1)
        return scores

    def get_embeddings(self):
        return self.user_emb.weight.detach().cpu().numpy(), self.item_emb.weight.detach().cpu().numpy()

def train_drip_v2(n_users, n_items, train_df, train_set, user_sequences, epochs=50):
    print("[DRIP v2] Training Improved Dynamic Relational Intent Propagation...")

    model = DRIPv2(n_users, n_items, CFG.DRIP_DIM, CFG.DRIP_HEADS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.DRIP_LR, weight_decay=1e-5)

    train_users = train_df['user'].values
    train_items = train_df['item'].values
    max_hist_len = 20

    losses = []
    for epoch in range(epochs):
        model.train()
        idx = np.random.permutation(len(train_users))
        total_loss, batch_size = 0, 512

        for i in range(0, len(idx), batch_size):
            batch_idx = idx[i:i+batch_size]

            users = torch.LongTensor(train_users[batch_idx]).to(DEVICE)
            pos_items = torch.LongTensor(train_items[batch_idx]).to(DEVICE)

            # Build histories
            histories, masks = [], []
            for u in train_users[batch_idx]:
                hist = user_sequences.get(u, [0])[-max_hist_len:]
                mask = [True] * len(hist)
                hist = hist + [0] * (max_hist_len - len(hist))
                mask = mask + [False] * (max_hist_len - len(mask))
                histories.append(hist)
                masks.append(mask)
            histories = torch.LongTensor(histories).to(DEVICE)
            masks = torch.BoolTensor(masks).to(DEVICE)

            # Negative sampling
            neg_items = []
            for u in train_users[batch_idx]:
                neg = np.random.randint(0, n_items)
                while (u, neg) in train_set:
                    neg = np.random.randint(0, n_items)
                neg_items.append(neg)
            neg_items = torch.LongTensor(neg_items).to(DEVICE)

            optimizer.zero_grad()

            pos_scores = model(users, pos_items, histories, masks)
            neg_scores = model(users, neg_items, histories, masks)

            loss = -F.logsigmoid(pos_scores - neg_scores).mean()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(idx) // batch_size)
        losses.append(avg_loss)

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_loss:.4f}")

    user_emb, item_emb = model.get_embeddings()
    print(f"  DRIP v2 embeddings: {user_emb.shape}, {item_emb.shape}")
    return model, user_emb, item_emb, losses

drip_model, drip_user_emb, drip_item_emb, drip_losses = train_drip_v2(
    n_users, n_items, train_df, train_set, user_sequences, epochs=CFG.DRIP_EPOCHS
)
training_history['DRIP'] = drip_losses

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# COMPONENT 2: MINT v2 (Improved Momentum-based Interest Network Tracking)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

print("\n" + "─" * 110)
print("COMPONENT 2: MINT v2 (Improved Momentum-based Interest Network Tracking)")
print("  Improvement: Better contrastive objective with hard negative mining")
print("─" * 110)

class MINTv2(nn.Module):
    """Improved MINT with proper contrastive learning."""
    def __init__(self, n_users, n_items, dim=64, momentum=0.995, temperature=0.1):
        super().__init__()
        self.n_users, self.n_items, self.dim = n_users, n_items, dim
        self.momentum, self.temperature = momentum, temperature

        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)

        self.user_emb_k = nn.Embedding(n_users, dim)
        self.item_emb_k = nn.Embedding(n_items, dim)

        self.projector = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

        self._copy_params()
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def _copy_params(self):
        self.user_emb_k.weight.data.copy_(self.user_emb.weight.data)
        self.item_emb_k.weight.data.copy_(self.item_emb.weight.data)
        for param in self.user_emb_k.parameters():
            param.requires_grad = False
        for param in self.item_emb_k.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def _momentum_update(self):
        for param_q, param_k in zip(self.user_emb.parameters(), self.user_emb_k.parameters()):
            param_k.data = param_k.data * self.momentum + param_q.data * (1. - self.momentum)
        for param_q, param_k in zip(self.item_emb.parameters(), self.item_emb_k.parameters()):
            param_k.data = param_k.data * self.momentum + param_q.data * (1. - self.momentum)

    def forward(self, users, pos_items, neg_items):
        # Query embeddings
        u_q = self.user_emb(users)
        i_q = self.item_emb(pos_items)
        q = F.normalize(self.projector(u_q + i_q), dim=-1)

        # Key embeddings (momentum)
        with torch.no_grad():
            self._momentum_update()
            u_k = self.user_emb_k(users)
            i_k = self.item_emb_k(pos_items)
            k_pos = F.normalize(self.projector(u_k + i_k), dim=-1)

            # Negative keys
            u_k_neg = self.user_emb_k(users)
            i_k_neg = self.item_emb_k(neg_items)
            k_neg = F.normalize(self.projector(u_k_neg + i_k_neg), dim=-1)

        # Contrastive loss
        l_pos = (q * k_pos).sum(dim=-1, keepdim=True) / self.temperature
        l_neg = (q * k_neg).sum(dim=-1, keepdim=True) / self.temperature

        logits = torch.cat([l_pos, l_neg], dim=1)
        labels = torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device)

        return F.cross_entropy(logits, labels)

    def get_embeddings(self):
        return self.user_emb.weight.detach().cpu().numpy(), self.item_emb.weight.detach().cpu().numpy()

def train_mint_v2(n_users, n_items, train_df, train_set, epochs=40):
    print("[MINT v2] Training Improved Momentum-based Interest Network Tracking...")

    model = MINTv2(n_users, n_items, CFG.MINT_DIM, CFG.MINT_MOMENTUM, CFG.MINT_TEMPERATURE).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.MINT_LR)

    train_users = train_df['user'].values
    train_items = train_df['item'].values

    losses = []
    for epoch in range(epochs):
        model.train()
        idx = np.random.permutation(len(train_users))
        total_loss, batch_size = 0, 512

        for i in range(0, len(idx), batch_size):
            batch_idx = idx[i:i+batch_size]
            users = torch.LongTensor(train_users[batch_idx]).to(DEVICE)
            pos_items = torch.LongTensor(train_items[batch_idx]).to(DEVICE)

            # Hard negative mining: random negatives
            neg_items = []
            for u in train_users[batch_idx]:
                neg = np.random.randint(0, n_items)
                while (u, neg) in train_set:
                    neg = np.random.randint(0, n_items)
                neg_items.append(neg)
            neg_items = torch.LongTensor(neg_items).to(DEVICE)

            optimizer.zero_grad()
            loss = model(users, pos_items, neg_items)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(idx) // batch_size)
        losses.append(avg_loss)

        if (epoch + 1) % 8 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_loss:.4f}")

    user_emb, item_emb = model.get_embeddings()
    print(f"  MINT v2 embeddings: {user_emb.shape}, {item_emb.shape}")
    return model, user_emb, item_emb, losses

mint_model, mint_user_emb, mint_item_emb, mint_losses = train_mint_v2(
    n_users, n_items, train_df, train_set, epochs=CFG.MINT_EPOCHS
)
training_history['MINT'] = mint_losses

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# COMPONENT 3: BRIDGE v2 (Already fixed, optimized training)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

print("\n" + "─" * 110)
print("COMPONENT 3: BRIDGE v2 (Bidirectional Relational Information Diffusion)")
print("  Improvement: Optimized bidirectional propagation with residual connections")
print("─" * 110)

class BRIDGEv2(nn.Module):
    """Improved BRIDGE with residual connections."""
    def __init__(self, n_users, n_items, dim=64, n_steps=3):
        super().__init__()
        self.n_users, self.n_items, self.dim, self.n_steps = n_users, n_items, dim, n_steps

        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)

        self.forward_transforms = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_steps)])
        self.backward_transforms = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_steps)])

        self.fusion = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

        self.alpha = nn.Parameter(torch.tensor(0.5))
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def propagate(self, emb, adj, transforms):
        h = emb
        outputs = [h]
        for transform in transforms:
            h_new = torch.sparse.mm(adj, h)
            h_new = transform(h_new)
            h = F.relu(h + h_new)  # Residual connection
            outputs.append(h)
        return torch.stack(outputs, dim=0).mean(dim=0)

    def forward(self, adj_norm, adj_norm_t):
        all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)

        forward_emb = self.propagate(all_emb, adj_norm, self.forward_transforms)
        backward_emb = self.propagate(all_emb, adj_norm_t, self.backward_transforms)

        alpha = torch.sigmoid(self.alpha)
        combined = torch.cat([alpha * forward_emb, (1 - alpha) * backward_emb], dim=-1)
        fused = self.fusion(combined)

        return fused[:self.n_users], fused[self.n_users:]

    def get_embeddings(self):
        return self.user_emb.weight.detach().cpu().numpy(), self.item_emb.weight.detach().cpu().numpy()

def train_bridge_v2(n_users, n_items, train_df, train_set, norm_adj, epochs=40):
    print("[BRIDGE v2] Training Bidirectional Diffusion...")

    coo = norm_adj.tocoo()
    adj_norm = torch.sparse_coo_tensor(torch.LongTensor(np.vstack([coo.row, coo.col])), torch.FloatTensor(coo.data), torch.Size(coo.shape)).to(DEVICE)
    adj_norm_t = torch.sparse_coo_tensor(torch.LongTensor(np.vstack([coo.col, coo.row])), torch.FloatTensor(coo.data), torch.Size(coo.shape)).to(DEVICE)

    model = BRIDGEv2(n_users, n_items, CFG.BRIDGE_DIM, CFG.BRIDGE_STEPS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.BRIDGE_LR)

    train_users = train_df['user'].values
    train_items = train_df['item'].values

    losses = []
    for epoch in range(epochs):
        model.train()
        idx = np.random.permutation(len(train_users))
        total_loss, batch_size = 0, 1024

        for i in range(0, len(idx), batch_size):
            batch_idx = idx[i:i+batch_size]
            users = torch.LongTensor(train_users[batch_idx]).to(DEVICE)
            pos_items = torch.LongTensor(train_items[batch_idx]).to(DEVICE)

            neg_items = []
            for u in train_users[batch_idx]:
                neg = np.random.randint(0, n_items)
                while (u, neg) in train_set:
                    neg = np.random.randint(0, n_items)
                neg_items.append(neg)
            neg_items = torch.LongTensor(neg_items).to(DEVICE)

            optimizer.zero_grad()
            user_emb, item_emb = model(adj_norm, adj_norm_t)

            u, pos, neg = user_emb[users], item_emb[pos_items], item_emb[neg_items]
            loss = -F.logsigmoid((u * pos).sum(dim=1) - (u * neg).sum(dim=1)).mean()
            reg = 0.001 * (model.user_emb.weight[users].norm(2).pow(2) + model.item_emb.weight[pos_items].norm(2).pow(2) + model.item_emb.weight[neg_items].norm(2).pow(2)) / len(users)
            loss = loss + reg

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(idx) // batch_size)
        losses.append(avg_loss)

        if (epoch + 1) % 8 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_loss:.4f}")

    model.eval()
    with torch.no_grad():
        user_emb, item_emb = model(adj_norm, adj_norm_t)

    print(f"  BRIDGE v2 embeddings: {user_emb.shape}, {item_emb.shape}")
    return model, user_emb.cpu().numpy(), item_emb.cpu().numpy(), losses

bridge_model, bridge_user_emb, bridge_item_emb, bridge_losses = train_bridge_v2(
    n_users, n_items, train_df, train_set, graph['norm_adj'], epochs=CFG.BRIDGE_EPOCHS
)
training_history['BRIDGE'] = bridge_losses

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# COMPONENT 4: NEXIS v2 (Improved Capsule Network)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

print("\n" + "─" * 110)
print("COMPONENT 4: NEXIS v2 (Improved Neural Expert Interest Synthesis)")
print("  Improvement: Simplified capsule with multi-interest extraction from history")
print("─" * 110)

class NEXISv2(nn.Module):
    """Improved NEXIS with multi-interest extraction."""
    def __init__(self, n_users, n_items, dim=64, n_interests=4):
        super().__init__()
        self.n_users, self.n_items, self.dim, self.n_interests = n_users, n_items, dim, n_interests

        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)

        # Multi-interest extraction
        self.interest_extractors = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))
            for _ in range(n_interests)
        ])

        # Attention over interests
        self.interest_attention = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Tanh(),
            nn.Linear(dim, 1)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.xavier_uniform_(m.weight)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, users, items):
        user_e = self.user_emb(users)
        item_e = self.item_emb(items)

        # Extract multiple interests from user embedding
        interests = []
        for extractor in self.interest_extractors:
            interest = extractor(user_e)
            interests.append(interest)
        interests = torch.stack(interests, dim=1)  # [B, K, D]

        # Attention: which interest is most relevant to target item?
        item_expanded = item_e.unsqueeze(1).expand(-1, self.n_interests, -1)  # [B, K, D]
        attn_input = torch.cat([interests, item_expanded], dim=-1)  # [B, K, 2D]
        attn_scores = self.interest_attention(attn_input).squeeze(-1)  # [B, K]
        attn_weights = F.softmax(attn_scores, dim=-1)  # [B, K]

        # Weighted sum of interests
        user_final = (interests * attn_weights.unsqueeze(-1)).sum(dim=1)  # [B, D]

        scores = (user_final * item_e).sum(dim=-1)
        return scores

    def get_embeddings(self):
        return self.user_emb.weight.detach().cpu().numpy(), self.item_emb.weight.detach().cpu().numpy()

def train_nexis_v2(n_users, n_items, train_df, train_set, epochs=40):
    print("[NEXIS v2] Training Improved Neural Expert Interest Synthesis...")

    model = NEXISv2(n_users, n_items, CFG.NEXIS_DIM, n_interests=4).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.NEXIS_LR)

    train_users = train_df['user'].values
    train_items = train_df['item'].values

    losses = []
    for epoch in range(epochs):
        model.train()
        idx = np.random.permutation(len(train_users))
        total_loss, batch_size = 0, 512

        for i in range(0, len(idx), batch_size):
            batch_idx = idx[i:i+batch_size]
            users = torch.LongTensor(train_users[batch_idx]).to(DEVICE)
            pos_items = torch.LongTensor(train_items[batch_idx]).to(DEVICE)

            neg_items = []
            for u in train_users[batch_idx]:
                neg = np.random.randint(0, n_items)
                while (u, neg) in train_set:
                    neg = np.random.randint(0, n_items)
                neg_items.append(neg)
            neg_items = torch.LongTensor(neg_items).to(DEVICE)

            optimizer.zero_grad()
            pos_scores = model(users, pos_items)
            neg_scores = model(users, neg_items)

            loss = -F.logsigmoid(pos_scores - neg_scores).mean()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(idx) // batch_size)
        losses.append(avg_loss)

        if (epoch + 1) % 8 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_loss:.4f}")

    user_emb, item_emb = model.get_embeddings()
    print(f"  NEXIS v2 embeddings: {user_emb.shape}, {item_emb.shape}")
    return model, user_emb, item_emb, losses

nexis_model, nexis_user_emb, nexis_item_emb, nexis_losses = train_nexis_v2(
    n_users, n_items, train_df, train_set, epochs=CFG.NEXIS_EPOCHS
)
training_history['NEXIS'] = nexis_losses

print("\nAll SYNAPSE-KG v2 components trained successfully!")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# COMPONENT 5: ORACLE v2 (Improved Meta-Learning Fusion)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

print("\n" + "─" * 110)
print("COMPONENT 5: ORACLE v2 (Improved Optimal Routing And Component-Level Ensembling)")
print("  Improvement: Deeper network with component-specific normalization")
print("─" * 110)

class ORACLEv2(nn.Module):
    """Improved ORACLE with deeper architecture."""
    def __init__(self, n_components, hidden_dim=64, init_temp=0.5):
        super().__init__()
        self.n_components = n_components

        self.score_normalizers = nn.ModuleList([
            nn.BatchNorm1d(1) for _ in range(n_components)
        ])

        self.encoder = nn.Sequential(
            nn.Linear(n_components, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_components)
        )

        self.temperature = nn.Parameter(torch.tensor(init_temp))
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, component_scores):
        # Normalize each component score separately
        normalized = []
        for i in range(self.n_components):
            score_i = component_scores[:, i:i+1]
            normalized.append(self.score_normalizers[i](score_i))
        normalized = torch.cat(normalized, dim=1)

        h = self.encoder(normalized)

        gate_logits = self.gate(h)
        gates = F.softmax(gate_logits / self.temperature.abs().clamp(min=0.1), dim=-1)

        weighted_scores = (normalized * gates).sum(dim=-1, keepdim=True)
        direct_out = self.output(h)

        return (weighted_scores + direct_out).squeeze(-1), gates

def build_oracle_v2_training_data(n_users, n_items, train_set, user_sequences,
                                   itemknn, features, drip_user_emb, drip_item_emb,
                                   mint_user_emb, mint_item_emb, bridge_user_emb, bridge_item_emb,
                                   nexis_user_emb, nexis_item_emb, lgcn_user_emb, lgcn_item_emb,
                                   n_pos=2000, n_neg_ratio=10):
    print("[ORACLE v2] Building training data...")

    rng = np.random.default_rng(CFG.SEED)

    pos_pairs = list(train_set)
    if len(pos_pairs) > n_pos:
        pos_pairs = rng.choice(pos_pairs, n_pos, replace=False).tolist()

    neg_pairs = []
    for u, i in pos_pairs:
        for _ in range(n_neg_ratio):
            neg = rng.integers(0, n_items)
            while (u, neg) in train_set:
                neg = rng.integers(0, n_items)
            neg_pairs.append((u, neg))

    all_pairs = [(u, i, 1) for u, i in pos_pairs] + [(u, i, 0) for u, i in neg_pairs]
    rng.shuffle(all_pairs)

    component_scores, labels = [], []

    for u, i, label in all_pairs:
        scores = [
            itemknn.predict(u, [i])[0],
            float(features['user_content'][u] @ features['item_content'][i]),
            features['item_popularity'][i],
            float(drip_user_emb[u] @ drip_item_emb[i]),
            float(mint_user_emb[u] @ mint_item_emb[i]),
            float(bridge_user_emb[u] @ bridge_item_emb[i]),
            float(nexis_user_emb[u] @ nexis_item_emb[i]),
            float(lgcn_user_emb[u] @ lgcn_item_emb[i])
        ]
        component_scores.append(scores)
        labels.append(label)

    print(f"  Training samples: {len(all_pairs)} ({n_pos} positive)")
    return np.array(component_scores), np.array(labels)

def train_oracle_v2(component_scores, labels, n_components=8, epochs=100):
    print("[ORACLE v2] Training meta-fusion...")

    model = ORACLEv2(n_components, hidden_dim=CFG.ORACLE_HIDDEN, init_temp=CFG.ORACLE_TEMP).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.ORACLE_LR)
    criterion = nn.BCEWithLogitsLoss()

    # Manual normalization
    scores_mean = component_scores.mean(axis=0)
    scores_std = component_scores.std(axis=0) + 1e-8
    scores_norm = (component_scores - scores_mean) / scores_std

    X = torch.FloatTensor(scores_norm).to(DEVICE)
    y = torch.FloatTensor(labels).to(DEVICE)

    dataset_size, batch_size = len(X), 256

    losses = []
    best_loss = float('inf')
    avg_gates = None

    for epoch in range(epochs):
        model.train()
        idx = torch.randperm(dataset_size)
        total_loss = 0
        all_gates = []

        for i in range(0, dataset_size, batch_size):
            batch_idx = idx[i:i+batch_size]
            batch_X, batch_y = X[batch_idx], y[batch_idx]

            optimizer.zero_grad()
            preds, gates = model(batch_X)
            loss = criterion(preds, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            all_gates.append(gates.detach())

        avg_loss = total_loss / max(1, dataset_size // batch_size)
        losses.append(avg_loss)

        if avg_loss < best_loss:
            best_loss = avg_loss
            avg_gates = torch.cat(all_gates, dim=0).mean(dim=0).cpu().numpy()

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_loss:.4f}, Temp={model.temperature.item():.3f}")

    print(f"  ORACLE v2 training complete. Best loss: {best_loss:.4f}")
    return model, scores_mean, scores_std, avg_gates, losses

oracle_X, oracle_y = build_oracle_v2_training_data(
    n_users, n_items, train_set, user_sequences, itemknn, features,
    drip_user_emb, drip_item_emb, mint_user_emb, mint_item_emb,
    bridge_user_emb, bridge_item_emb, nexis_user_emb, nexis_item_emb,
    lgcn_user_emb, lgcn_item_emb
)

oracle_model, oracle_mean, oracle_std, oracle_weights, oracle_losses = train_oracle_v2(
    oracle_X, oracle_y, n_components=8, epochs=CFG.ORACLE_EPOCHS
)
training_history['ORACLE'] = oracle_losses

component_names = ['ItemKNN', 'Content', 'Popularity', 'DRIP', 'MINT', 'BRIDGE', 'NEXIS', 'LightGCN']
print("\n  Learned Component Weights:")
for name, weight in zip(component_names, oracle_weights):
    print(f"    {name}: {weight:.4f}")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 6-12: EVALUATION, RESULTS, VISUALIZATION (same structure as before)
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("PHASE 6: SYNAPSE-KG v2 FINAL SCORING")
print("═" * 110)

def synapse_kg_v2_score(user, items, itemknn, features,
                         drip_user_emb, drip_item_emb,
                         mint_user_emb, mint_item_emb,
                         bridge_user_emb, bridge_item_emb,
                         nexis_user_emb, nexis_item_emb,
                         lgcn_user_emb, lgcn_item_emb,
                         oracle_model, oracle_mean, oracle_std):

    component_scores = []
    for item in items:
        scores = [
            itemknn.predict(user, [item])[0],
            float(features['user_content'][user] @ features['item_content'][item]),
            features['item_popularity'][item],
            float(drip_user_emb[user] @ drip_item_emb[item]),
            float(mint_user_emb[user] @ mint_item_emb[item]),
            float(bridge_user_emb[user] @ bridge_item_emb[item]),
            float(nexis_user_emb[user] @ nexis_item_emb[item]),
            float(lgcn_user_emb[user] @ lgcn_item_emb[item])
        ]
        component_scores.append(scores)

    component_scores = np.array(component_scores)
    scores_norm = (component_scores - oracle_mean) / oracle_std

    oracle_model.eval()
    with torch.no_grad():
        X = torch.FloatTensor(scores_norm).to(DEVICE)
        final_scores, _ = oracle_model(X)
        final_scores = final_scores.cpu().numpy()

    return final_scores

print("[SYNAPSE-KG v2] Final scoring function ready.")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 7: COMPREHENSIVE EVALUATION
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("PHASE 7: COMPREHENSIVE EVALUATION")
print("═" * 110)

def evaluate_model(score_fn, test_df, train_set, n_items, top_ks=[5, 10, 20]):
    metrics = {f'HR@{k}': [] for k in top_ks}
    metrics.update({f'NDCG@{k}': [] for k in top_ks})
    metrics['MRR@10'] = []

    for _, row in test_df.iterrows():
        user, true_item = int(row['user']), int(row['item'])

        neg_items = []
        while len(neg_items) < CFG.N_NEG:
            neg = np.random.randint(0, n_items)
            if (user, neg) not in train_set and neg != true_item:
                neg_items.append(neg)

        candidates = [true_item] + neg_items
        scores = score_fn(user, candidates)

        ranked_indices = np.argsort(-scores)
        rank = np.where(ranked_indices == 0)[0][0] + 1

        for k in top_ks:
            metrics[f'HR@{k}'].append(1.0 if rank <= k else 0.0)
            metrics[f'NDCG@{k}'].append(1.0 / np.log2(rank + 1) if rank <= k else 0.0)

        metrics['MRR@10'].append(1.0 / rank if rank <= 10 else 0.0)

    return {k: np.mean(v) for k, v in metrics.items()}

print("[Eval] Evaluating all models...")

results = {}

def pop_score(user, items):
    return np.array([item_popularity[i] for i in items])

def iknn_score(user, items):
    return itemknn.predict(user, items)

def uknn_score(user, items):
    return userknn.predict(user, items)

def content_score(user, items):
    return np.array([features['user_content'][user] @ features['item_content'][i] for i in items])

def mf_score(user, items):
    return np.array([mf_user_emb[user] @ mf_item_emb[i] for i in items])

def lgcn_score(user, items):
    return np.array([lgcn_user_emb[user] @ lgcn_item_emb[i] for i in items])

def drip_score(user, items):
    return np.array([drip_user_emb[user] @ drip_item_emb[i] for i in items])

def mint_score(user, items):
    return np.array([mint_user_emb[user] @ mint_item_emb[i] for i in items])

def bridge_score(user, items):
    return np.array([bridge_user_emb[user] @ bridge_item_emb[i] for i in items])

def nexis_score(user, items):
    return np.array([nexis_user_emb[user] @ nexis_item_emb[i] for i in items])

def synapse_score(user, items):
    return synapse_kg_v2_score(user, items, itemknn, features,
                               drip_user_emb, drip_item_emb,
                               mint_user_emb, mint_item_emb,
                               bridge_user_emb, bridge_item_emb,
                               nexis_user_emb, nexis_item_emb,
                               lgcn_user_emb, lgcn_item_emb,
                               oracle_model, oracle_mean, oracle_std)

# Evaluate all models
for name, fn in [('Popularity', pop_score), ('ItemKNN', iknn_score), ('UserKNN', uknn_score),
                 ('Content', content_score), ('MF-BPR', mf_score), ('LightGCN', lgcn_score),
                 ('DRIP', drip_score), ('MINT', mint_score), ('BRIDGE', bridge_score),
                 ('NEXIS', nexis_score), ('SYNAPSE-KG', synapse_score)]:
    start = time.time()
    results[name] = evaluate_model(fn, test_df, train_set, n_items)
    print(f"  {name}: NDCG@10={results[name]['NDCG@10']:.4f}, HR@10={results[name]['HR@10']:.4f} ({time.time()-start:.1f}s)")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 8: RESULTS DISPLAY
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("RESULTS")
print("═" * 110)

sorted_models = sorted(results.keys(), key=lambda x: results[x]['NDCG@10'])

print(f"\n{'Model':<20} {'HR@5':>8} {'HR@10':>8} {'HR@20':>8} {'NDCG@10':>10} {'MRR@10':>10}")
print("─" * 75)

for model in sorted_models:
    r = results[model]
    prefix = "★" if model == 'SYNAPSE-KG' else " "
    print(f"{prefix}{model:<19} {r['HR@5']:>8.4f} {r['HR@10']:>8.4f} {r['HR@20']:>8.4f} {r['NDCG@10']:>10.4f} {r['MRR@10']:>10.4f}")

baselines = ['Popularity', 'ItemKNN', 'UserKNN', 'Content', 'MF-BPR', 'LightGCN']
best_baseline = max(baselines, key=lambda x: results[x]['NDCG@10'])
best_baseline_ndcg = results[best_baseline]['NDCG@10']
synapse_ndcg = results['SYNAPSE-KG']['NDCG@10']
improvement = (synapse_ndcg - best_baseline_ndcg) / best_baseline_ndcg * 100

print("\n" + "─" * 75)
print(f"Best Baseline: {best_baseline} (NDCG@10: {best_baseline_ndcg:.4f})")
print(f"SYNAPSE-KG v2: NDCG@10 = {synapse_ndcg:.4f} ({improvement:+.2f}%)")

results_df = pd.DataFrame([{'Model': m, **results[m]} for m in sorted_models])
results_df.to_csv(f"{CFG.OUTPUT_PATH}/tables/Table3_Main_Results.csv", index=False)
print("\n[Saved] Table 3: Main Results")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 9-12: ABLATION, STATS, VISUALIZATION (abbreviated for space)
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("ABLATION STUDY")
print("═" * 110)

ablation = {'SYNAPSE-KG (Full)': results['SYNAPSE-KG']}

def make_ablation_fn(exclude_idx):
    def fn(user, items):
        component_scores = []
        for item in items:
            scores = [
                itemknn.predict(user, [item])[0],
                float(features['user_content'][user] @ features['item_content'][item]),
                features['item_popularity'][item],
                float(drip_user_emb[user] @ drip_item_emb[item]) if exclude_idx != 3 else 0.0,
                float(mint_user_emb[user] @ mint_item_emb[item]) if exclude_idx != 4 else 0.0,
                float(bridge_user_emb[user] @ bridge_item_emb[item]) if exclude_idx != 5 else 0.0,
                float(nexis_user_emb[user] @ nexis_item_emb[item]) if exclude_idx != 6 else 0.0,
                float(lgcn_user_emb[user] @ lgcn_item_emb[item])
            ]
            component_scores.append(scores)
        component_scores = np.array(component_scores)
        scores_norm = (component_scores - oracle_mean) / oracle_std
        oracle_model.eval()
        with torch.no_grad():
            X = torch.FloatTensor(scores_norm).to(DEVICE)
            final_scores, _ = oracle_model(X)
        return final_scores.cpu().numpy()
    return fn

for name, idx in [('w/o DRIP', 3), ('w/o MINT', 4), ('w/o BRIDGE', 5), ('w/o NEXIS', 6)]:
    ablation[name] = evaluate_model(make_ablation_fn(idx), test_df, train_set, n_items)
    print(f"  {name}: NDCG@10={ablation[name]['NDCG@10']:.4f}")

ablation_df = pd.DataFrame([{'Variant': v, **ablation[v]} for v in ablation])
ablation_df.to_csv(f"{CFG.OUTPUT_PATH}/tables/Table5_Ablation_Study.csv", index=False)
print("[Saved] Table 5: Ablation Study")

# Statistical tests
print("\n" + "═" * 110)
print("STATISTICAL SIGNIFICANCE TESTING")
print("═" * 110)

def compute_per_user_metrics(score_fn, test_df, train_set, n_items):
    ndcg_scores = []
    for _, row in test_df.iterrows():
        user, true_item = int(row['user']), int(row['item'])
        neg_items = []
        while len(neg_items) < CFG.N_NEG:
            neg = np.random.randint(0, n_items)
            if (user, neg) not in train_set and neg != true_item:
                neg_items.append(neg)
        candidates = [true_item] + neg_items
        scores = score_fn(user, candidates)
        ranked_indices = np.argsort(-scores)
        rank = np.where(ranked_indices == 0)[0][0] + 1
        ndcg_scores.append(1.0 / np.log2(rank + 1) if rank <= 10 else 0.0)
    return np.array(ndcg_scores)

print("[Stats] Computing per-user metrics...")
synapse_per_user = compute_per_user_metrics(synapse_score, test_df, train_set, n_items)
mf_per_user = compute_per_user_metrics(mf_score, test_df, train_set, n_items)
lgcn_per_user = compute_per_user_metrics(lgcn_score, test_df, train_set, n_items)
itemknn_per_user = compute_per_user_metrics(iknn_score, test_df, train_set, n_items)

stat_results = []
print(f"\n{'Comparison':<30} {'t-stat':>10} {'p-value':>12} {'Cohen d':>10} {'Sig':>5}")
print("─" * 75)

for name1, name2, scores1, scores2 in [('SYNAPSE-KG', 'MF-BPR', synapse_per_user, mf_per_user),
                                        ('SYNAPSE-KG', 'LightGCN', synapse_per_user, lgcn_per_user),
                                        ('SYNAPSE-KG', 'ItemKNN', synapse_per_user, itemknn_per_user)]:
    diff = scores1 - scores2
    t_stat, p_value = stats.ttest_rel(scores1, scores2)
    pooled_std = np.sqrt((scores1.std()**2 + scores2.std()**2) / 2)
    cohens_d = diff.mean() / pooled_std if pooled_std > 0 else 0
    sig = "✓" if p_value < 0.05 else ""
    print(f"{name1} vs {name2:<15} {t_stat:>10.3f} {p_value:>12.2e} {cohens_d:>10.3f} {sig:>5}")
    stat_results.append({'Comparison': f'{name1} vs {name2}', 't-statistic': t_stat, 'p-value': p_value, 'Cohen_d': cohens_d, 'Significant': p_value < 0.05})

stat_df = pd.DataFrame(stat_results)
stat_df.to_csv(f"{CFG.OUTPUT_PATH}/tables/Table4_Statistical_Tests.csv", index=False)
print("\n[Saved] Table 4: Statistical Significance Tests")

# Generate figures
print("\n" + "═" * 110)
print("GENERATING FIGURES")
print("═" * 110)

# Figure 1: NDCG@10 Comparison
fig1, ax1 = plt.subplots(figsize=(12, 6))
ndcg_vals = [results[m]['NDCG@10'] for m in sorted_models]
colors = ['#27ae60' if m == 'SYNAPSE-KG' else '#3498db' if m in ['DRIP', 'MINT', 'BRIDGE', 'NEXIS'] else '#95a5a6' for m in sorted_models]
bars = ax1.barh(sorted_models, ndcg_vals, color=colors, edgecolor='black', linewidth=0.5)
ax1.axvline(best_baseline_ndcg, color='#e74c3c', linestyle='--', linewidth=2, label=f'Best Baseline ({best_baseline})')
ax1.set_xlabel('NDCG@10', fontsize=12, fontweight='bold')
ax1.set_title('Model Comparison: NDCG@10', fontsize=14, fontweight='bold')
ax1.legend()
plt.tight_layout()
fig1.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure1_NDCG10_Comparison.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig1)
print("[Saved] Figure 1: NDCG@10 Comparison")

# Figure 5: Component Weights
fig5, ax5 = plt.subplots(figsize=(10, 6))
weight_colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(component_names)))
bars = ax5.bar(component_names, oracle_weights, color=weight_colors, edgecolor='black')
ax5.set_ylabel('Weight', fontsize=12, fontweight='bold')
ax5.set_title('ORACLE v2 Component Weights (Learned)', fontsize=14, fontweight='bold')
ax5.tick_params(axis='x', rotation=45)
plt.tight_layout()
fig5.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure5_Component_Weights.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig5)
print("[Saved] Figure 5: Component Weights")

# Executive Summary
print("\n" + "═" * 110)
print("SYNAPSE-KG v2 EXECUTIVE SUMMARY")
print("═" * 110)

summary = f"""
╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║  SYNAPSE-KG v2: Improved Knowledge Graph Enhanced Recommendations                                               ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

📊 PERFORMANCE RESULTS
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  SYNAPSE-KG v2:
    HR@5:  {results['SYNAPSE-KG']['HR@5']:.4f} | HR@10:  {results['SYNAPSE-KG']['HR@10']:.4f} | HR@20:  {results['SYNAPSE-KG']['HR@20']:.4f}
    NDCG@10: {results['SYNAPSE-KG']['NDCG@10']:.4f} | MRR@10: {results['SYNAPSE-KG']['MRR@10']:.4f}

  Best External Baseline: {best_baseline} (NDCG@10: {best_baseline_ndcg:.4f})

  ═══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  ★ IMPROVEMENT: {improvement:+.2f}% over best baseline ★
  ═══════════════════════════════════════════════════════════════════════════════════════════════════════════════

⚙️ ORACLE v2 COMPONENT WEIGHTS
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  {', '.join([f'{n}: {w:.3f}' for n, w in zip(component_names, oracle_weights)])}

════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
"""
print(summary)

with open(f'{CFG.OUTPUT_PATH}/tables/Summary.txt', 'w') as f:
    f.write(summary)

print("\n" + "═" * 110)
print("✓ SYNAPSE-KG v2 pipeline completed successfully!")
print("═" * 110)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  SYNAPSE-KG v2: Improved Knowledge Graph Enhanced Recommendations
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Device: cpu | PyTorch: 2.9.0+cpu
  Output: ./SYNAPSE-KG-v2-Results
══════════════════════════════════════════════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────────────────────────────────────────────
PHASE 1: DATA LOADING
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
[Data] Loaded: 1,000,209 ratings, 3,883 movies, 6,040 users

──────────────────────────────────────────────────────────────────────────────────────────────────────────────
PHASE 1.5: DATA PREPROCESSING
────────────────────────────────────────────────────────────────────────────────────

In [3]:
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# SYNAPSE-KG: Semantic-Yielding Neural Attention with Path Synthesis and Encoding
#             for Knowledge Graph Enhanced Recommendations
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
#
# NOVEL COMPONENTS:
# 1. DRIP: Dynamic Relational Intent Propagation - User intent-aware message passing
# 2. MINT: Momentum-based Interest Network Tracking - Momentum contrast for preferences
# 3. BRIDGE: Bidirectional Relational Information Diffusion with Graph Encoding
# 4. NEXIS: Neural EXpert Interest Synthesis - Multi-interest extraction
# 5. ORACLE: Optimal Routing And Component-Level Ensembling - Meta-learning fusion
#
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

import os
import sys
import io
import math
import random
import warnings
import zipfile
import urllib.request
import time
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Set, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy import stats, sparse as sp
from scipy.sparse import csr_matrix, coo_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    """Configuration for SYNAPSE-KG model."""
    # Paths
    OUTPUT_PATH: str = "/content/drive/MyDrive/SYNAPSE/SYNAPSE-KG-Results+"
    ML_ROOT: str = "./data/ml-1m"
    ML_URL: str = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

    # Data settings
    SEED: int = 42
    SAMPLE_PCT: float = 0.35
    MIN_USER_INTER: int = 10
    MIN_ITEM_INTER: int = 5
    N_NEG: int = 99
    TOP_KS: List[int] = field(default_factory=lambda: [5, 10, 20])

    # DRIP settings
    DRIP_DIM: int = 64
    DRIP_LAYERS: int = 2
    DRIP_HEADS: int = 4
    DRIP_EPOCHS: int = 50
    DRIP_LR: float = 5e-4

    # MINT settings
    MINT_DIM: int = 64
    MINT_MOMENTUM: float = 0.995
    MINT_EPOCHS: int = 40
    MINT_LR: float = 5e-4
    MINT_QUEUE_SIZE: int = 2048
    MINT_TEMPERATURE: float = 0.1

    # BRIDGE settings
    BRIDGE_DIM: int = 64
    BRIDGE_STEPS: int = 3
    BRIDGE_EPOCHS: int = 40
    BRIDGE_LR: float = 1e-3

    # NEXIS settings
    NEXIS_DIM: int = 64
    NEXIS_CAPSULES: int = 8
    NEXIS_ROUTING_ITERS: int = 3
    NEXIS_EPOCHS: int = 40
    NEXIS_LR: float = 5e-4

    # ORACLE settings
    ORACLE_EPOCHS: int = 100
    ORACLE_LR: float = 1e-3
    ORACLE_TEMP: float = 0.5
    ORACLE_HIDDEN: int = 64

    # Baseline settings
    LGCN_DIM: int = 64
    LGCN_LAYERS: int = 3
    LGCN_EPOCHS: int = 30
    LGCN_LR: float = 1e-3

    MF_EPOCHS: int = 40
    MF_DIM: int = 64
    MF_LR: float = 1e-3


CFG = Config()

# Create directories
os.makedirs(CFG.OUTPUT_PATH, exist_ok=True)
os.makedirs(f"{CFG.OUTPUT_PATH}/figures", exist_ok=True)
os.makedirs(f"{CFG.OUTPUT_PATH}/tables", exist_ok=True)
os.makedirs(CFG.ML_ROOT, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CFG.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("═" * 110)
print("  SYNAPSE-KG: Semantic-Yielding Neural Attention with Path Synthesis and Encoding")
print("             for Knowledge Graph Enhanced Recommendations")
print("═" * 110)
print(f"  Device: {DEVICE} | PyTorch: {torch.__version__}")
print(f"  Output: {CFG.OUTPUT_PATH}")
print("═" * 110)

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 1: DATA LOADING & PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 110)
print("PHASE 1: DATA LOADING")
print("─" * 110)

def download_movielens():
    ratings_path = os.path.join(CFG.ML_ROOT, "ratings.dat")
    if not os.path.exists(ratings_path):
        print("[Data] Downloading MovieLens-1M...")
        with urllib.request.urlopen(CFG.ML_URL, timeout=120) as response:
            data = response.read()
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            for name in zf.namelist():
                if name.endswith('.dat'):
                    with zf.open(name) as src:
                        dest_path = os.path.join(CFG.ML_ROOT, os.path.basename(name))
                        with open(dest_path, 'wb') as dst:
                            dst.write(src.read())
        print("[Data] Download complete.")

def load_movielens():
    download_movielens()
    ratings = pd.read_csv(f"{CFG.ML_ROOT}/ratings.dat", sep="::", names=["userId", "movieId", "rating", "timestamp"], engine="python", encoding="latin-1")
    movies = pd.read_csv(f"{CFG.ML_ROOT}/movies.dat", sep="::", names=["movieId", "title", "genres"], engine="python", encoding="latin-1")
    return ratings, movies

ratings_df, movies_df = load_movielens()
print(f"[Data] Loaded: {len(ratings_df):,} ratings, {len(movies_df):,} movies, {ratings_df['userId'].nunique():,} users")

print("\n" + "─" * 110)
print("PHASE 1.5: DATA PREPROCESSING")
print("─" * 110)

def preprocess_data(ratings_df, movies_df):
    df = ratings_df[ratings_df['rating'] >= 4][['userId', 'movieId', 'timestamp', 'rating']].copy()

    for _ in range(3):
        user_counts = df.groupby('userId').size()
        df = df[df['userId'].isin(user_counts[user_counts >= CFG.MIN_USER_INTER].index)]
        item_counts = df.groupby('movieId').size()
        df = df[df['movieId'].isin(item_counts[item_counts >= CFG.MIN_ITEM_INTER].index)]

    rng = np.random.default_rng(CFG.SEED)
    all_users = df['userId'].unique()
    sampled_users = rng.choice(all_users, int(len(all_users) * CFG.SAMPLE_PCT), replace=False)
    df = df[df['userId'].isin(sampled_users)]

    user_counts = df.groupby('userId').size()
    df = df[df['userId'].isin(user_counts[user_counts >= CFG.MIN_USER_INTER].index)]

    users = sorted(df['userId'].unique())
    items = sorted(df['movieId'].unique())
    user_to_idx = {u: i for i, u in enumerate(users)}
    item_to_idx = {m: i for i, m in enumerate(items)}
    n_users, n_items = len(users), len(items)

    df['user'] = df['userId'].map(user_to_idx)
    df['item'] = df['movieId'].map(item_to_idx)
    df = df.sort_values(['user', 'timestamp']).reset_index(drop=True)

    user_sequences = {}
    for u in range(n_users):
        user_df = df[df['user'] == u].sort_values('timestamp')
        user_sequences[u] = user_df['item'].tolist()

    train_records, test_records = [], []
    for u in range(n_users):
        user_df = df[df['user'] == u].sort_values('timestamp')
        if len(user_df) >= 2:
            test_records.append(user_df.iloc[-1])
            train_records.extend(user_df.iloc[:-1].to_dict('records'))

    train_df = pd.DataFrame(train_records)
    test_df = pd.DataFrame(test_records)

    movies_subset = movies_df[movies_df['movieId'].isin(item_to_idx)].copy()
    movies_subset['item'] = movies_subset['movieId'].map(item_to_idx)

    train_set = set(zip(train_df['user'], train_df['item']))

    return {
        'train_df': train_df, 'test_df': test_df, 'movies': movies_subset,
        'n_users': n_users, 'n_items': n_items, 'user_to_idx': user_to_idx,
        'item_to_idx': item_to_idx, 'user_sequences': user_sequences, 'train_set': train_set
    }

data = preprocess_data(ratings_df, movies_df)
train_df, test_df, movies_subset = data['train_df'], data['test_df'], data['movies']
n_users, n_items = data['n_users'], data['n_items']
user_sequences, train_set = data['user_sequences'], data['train_set']

print(f"  Users: {n_users:,}, Items: {n_items:,}")
print(f"  Train: {len(train_df):,}, Test: {len(test_df):,}")
print(f"  Sparsity: {(1 - len(train_df) / (n_users * n_items)) * 100:.2f}%")

# Save Table 1
table1 = pd.DataFrame({
    'Metric': ['Users', 'Items', 'Train Interactions', 'Test Interactions', 'Sparsity (%)', 'Avg Items/User'],
    'Value': [n_users, n_items, len(train_df), len(test_df), f"{(1 - len(train_df) / (n_users * n_items)) * 100:.2f}", f"{len(train_df) / n_users:.1f}"]
})
table1.to_csv(f"{CFG.OUTPUT_PATH}/tables/Table1_Dataset_Statistics.csv", index=False)
print("[Saved] Table 1: Dataset Statistics")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 2: KNOWLEDGE GRAPH CONSTRUCTION
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 110)
print("PHASE 2: KNOWLEDGE GRAPH CONSTRUCTION")
print("─" * 110)

def build_knowledge_graph(train_df, movies_subset, n_users, n_items, user_sequences):
    triples = []
    rel_counts = defaultdict(int)

    USER_OFFSET, ITEM_OFFSET = 0, n_users

    all_genres = set()
    for genres in movies_subset['genres']:
        all_genres.update(genres.split('|'))
    genre_to_idx = {g: i for i, g in enumerate(sorted(all_genres))}
    n_genres = len(genre_to_idx)
    GENRE_OFFSET = ITEM_OFFSET + n_items

    for _, row in train_df.iterrows():
        u, i = int(row['user']), int(row['item'])
        triples.append((USER_OFFSET + u, 0, ITEM_OFFSET + i))
        rel_counts['INTERACTED'] += 1

    for _, row in movies_subset.iterrows():
        item_idx = int(row['item'])
        for genre in row['genres'].split('|'):
            if genre in genre_to_idx:
                triples.append((ITEM_OFFSET + item_idx, 1, GENRE_OFFSET + genre_to_idx[genre]))
                rel_counts['HAS_GENRE'] += 1

    co_watch_count = 0
    for u, items in user_sequences.items():
        if len(items) >= 2:
            for i in range(len(items)):
                for j in range(i + 1, min(i + 6, len(items))):
                    if co_watch_count < 25000:
                        triples.append((ITEM_OFFSET + items[i], 2, ITEM_OFFSET + items[j]))
                        co_watch_count += 1
    rel_counts['CO_WATCHED'] = co_watch_count

    item_genres = {int(row['item']): set(row['genres'].split('|')) for _, row in movies_subset.iterrows()}
    genre_sim_count = 0
    items_list = list(item_genres.keys())
    rng = np.random.default_rng(CFG.SEED)
    rng.shuffle(items_list)
    for i, item1 in enumerate(items_list[:500]):
        for item2 in items_list[i+1:i+50]:
            if item1 in item_genres and item2 in item_genres:
                if len(item_genres[item1] & item_genres[item2]) >= 2 and genre_sim_count < 10000:
                    triples.append((ITEM_OFFSET + item1, 3, ITEM_OFFSET + item2))
                    genre_sim_count += 1
    rel_counts['GENRE_SIMILAR'] = genre_sim_count

    temporal_count = 0
    for u, items in user_sequences.items():
        for i in range(len(items) - 1):
            triples.append((ITEM_OFFSET + items[i], 4, ITEM_OFFSET + items[i + 1]))
            temporal_count += 1
    rel_counts['TEMPORAL_NEXT'] = temporal_count

    return {
        'triples': triples, 'n_entities': GENRE_OFFSET + n_genres, 'n_relations': 5,
        'rel_counts': dict(rel_counts), 'USER_OFFSET': USER_OFFSET, 'ITEM_OFFSET': ITEM_OFFSET,
        'GENRE_OFFSET': GENRE_OFFSET, 'n_genres': n_genres, 'genre_to_idx': genre_to_idx
    }

kg = build_knowledge_graph(train_df, movies_subset, n_users, n_items, user_sequences)
print(f"  Entities: {kg['n_entities']:,}, Triples: {len(kg['triples']):,}")
print(f"  Relations: {kg['rel_counts']}")

table2 = pd.DataFrame([{'Relation': rel, 'Count': count} for rel, count in kg['rel_counts'].items()])
table2.to_csv(f"{CFG.OUTPUT_PATH}/tables/Table2_KG_Statistics.csv", index=False)
print("[Saved] Table 2: KG Statistics")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 3: FEATURE ENGINEERING & GRAPH STRUCTURES
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 110)
print("PHASE 3: FEATURE ENGINEERING")
print("─" * 110)

def build_features(train_df, movies_subset, n_users, n_items, user_sequences):
    print("[Features] Building content embeddings...")
    item_texts = []
    for i in range(n_items):
        row = movies_subset[movies_subset['item'] == i]
        text = row.iloc[0]['genres'].replace('|', ' ') + ' ' + row.iloc[0]['title'] if len(row) > 0 else 'unknown'
        item_texts.append(text)

    tfidf = TfidfVectorizer(max_features=500, stop_words='english')
    item_tfidf = tfidf.fit_transform(item_texts)
    svd = TruncatedSVD(n_components=64, random_state=CFG.SEED)
    item_content_emb = normalize(svd.fit_transform(item_tfidf))

    user_content_emb = np.zeros((n_users, 64))
    for u in range(n_users):
        items = user_sequences.get(u, [])
        if items:
            user_content_emb[u] = item_content_emb[items].mean(axis=0)
    user_content_emb = normalize(user_content_emb)

    print(f"  Content embeddings: users={user_content_emb.shape}, items={item_content_emb.shape}")

    print("[Features] Building popularity features...")
    item_pop = train_df.groupby('item').size().reindex(range(n_items), fill_value=0).values
    item_pop_norm = (item_pop - item_pop.min()) / (item_pop.max() - item_pop.min() + 1e-8)

    return {'user_content': user_content_emb, 'item_content': item_content_emb, 'item_popularity': item_pop_norm}

features = build_features(train_df, movies_subset, n_users, n_items, user_sequences)

print("\n" + "─" * 110)
print("PHASE 3.5: BUILDING GRAPH STRUCTURES")
print("─" * 110)

def build_graph_structures(train_df, n_users, n_items):
    rows = train_df['user'].values
    cols = train_df['item'].values + n_users
    data = np.ones(len(train_df))
    n_nodes = n_users + n_items
    adj = csr_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes))
    adj = adj + adj.T
    degrees = np.array(adj.sum(axis=1)).flatten()
    d_inv_sqrt = np.power(degrees + 1e-8, -0.5)
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    norm_adj = d_mat_inv_sqrt @ adj @ d_mat_inv_sqrt
    print(f"  Adjacency matrix: {adj.nnz:,} edges")
    return {'adj': adj, 'norm_adj': norm_adj, 'n_nodes': n_nodes}

graph = build_graph_structures(train_df, n_users, n_items)

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 4: EXTERNAL BASELINES
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("PHASE 4: EXTERNAL BASELINES")
print("═" * 110)

# ItemKNN
print("\n[Baseline 1] ItemKNN...")
class ItemKNN:
    def __init__(self, k=100):
        self.k = k
    def fit(self, train_df, n_users, n_items):
        self.n_users, self.n_items = n_users, n_items
        rows, cols = train_df['item'].values, train_df['user'].values
        item_user = csr_matrix((np.ones(len(train_df)), (rows, cols)), shape=(n_items, n_users))
        self.sim = normalize(item_user, axis=1) @ normalize(item_user, axis=1).T
        self.user_items = defaultdict(set)
        for _, row in train_df.iterrows():
            self.user_items[int(row['user'])].add(int(row['item']))
        return self
    def predict(self, user, items):
        history = list(self.user_items[user])
        if not history:
            return np.zeros(len(items))
        scores = []
        for item in items:
            sim_scores = self.sim[item, history].toarray().flatten()
            top_k = min(self.k, len(sim_scores))
            scores.append(np.partition(sim_scores, -top_k)[-top_k:].sum())
        return np.array(scores)

itemknn = ItemKNN(k=100).fit(train_df, n_users, n_items)
print("  ItemKNN ready.")

# UserKNN
print("[Baseline 2] UserKNN...")
class UserKNN:
    def __init__(self, k=50):
        self.k = k
    def fit(self, train_df, n_users, n_items):
        self.n_users, self.n_items = n_users, n_items
        rows, cols = train_df['user'].values, train_df['item'].values
        self.user_item = csr_matrix((np.ones(len(train_df)), (rows, cols)), shape=(n_users, n_items))
        self.sim = normalize(self.user_item, axis=1) @ normalize(self.user_item, axis=1).T
        return self
    def predict(self, user, items):
        sim_users = self.sim[user].toarray().flatten()
        top_k_idx = np.argsort(sim_users)[-self.k-1:-1]
        top_k_sim = sim_users[top_k_idx]
        scores = []
        for item in items:
            item_ratings = self.user_item[top_k_idx, item].toarray().flatten()
            scores.append((top_k_sim * item_ratings).sum() / (np.abs(top_k_sim).sum() + 1e-8))
        return np.array(scores)

userknn = UserKNN(k=50).fit(train_df, n_users, n_items)
print("  UserKNN ready.")

# MF-BPR
print("[Baseline 3] MF-BPR...")
class MFBPR(nn.Module):
    def __init__(self, n_users, n_items, dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
    def forward(self, users, pos_items, neg_items):
        u, pos, neg = self.user_emb(users), self.item_emb(pos_items), self.item_emb(neg_items)
        pos_scores = (u * pos).sum(dim=1)
        neg_scores = (u * neg).sum(dim=1)
        loss = -F.logsigmoid(pos_scores - neg_scores).mean()
        reg = 0.001 * (u.norm(2).pow(2) + pos.norm(2).pow(2) + neg.norm(2).pow(2)) / len(users)
        return loss + reg
    def get_embeddings(self):
        return self.user_emb.weight.detach().cpu().numpy(), self.item_emb.weight.detach().cpu().numpy()

def train_mf_bpr(n_users, n_items, train_df, train_set, epochs=40, dim=64, lr=1e-3):
    model = MFBPR(n_users, n_items, dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_users, train_items = train_df['user'].values, train_df['item'].values
    for epoch in range(epochs):
        model.train()
        idx = np.random.permutation(len(train_users))
        total_loss, batch_size = 0, 1024
        for i in range(0, len(idx), batch_size):
            batch_idx = idx[i:i+batch_size]
            users = torch.LongTensor(train_users[batch_idx]).to(DEVICE)
            pos_items = torch.LongTensor(train_items[batch_idx]).to(DEVICE)
            neg_items = []
            for u in train_users[batch_idx]:
                neg = np.random.randint(0, n_items)
                while (u, neg) in train_set:
                    neg = np.random.randint(0, n_items)
                neg_items.append(neg)
            neg_items = torch.LongTensor(neg_items).to(DEVICE)
            optimizer.zero_grad()
            loss = model(users, pos_items, neg_items)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={total_loss/(len(idx)//batch_size):.4f}")
    return model

mf_model = train_mf_bpr(n_users, n_items, train_df, train_set, epochs=CFG.MF_EPOCHS)
mf_user_emb, mf_item_emb = mf_model.get_embeddings()
print("  MF-BPR ready.")

# LightGCN
print("[Baseline 4] LightGCN...")
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, dim=64, n_layers=3):
        super().__init__()
        self.n_users, self.n_items, self.n_layers = n_users, n_items, n_layers
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
    def forward(self, norm_adj):
        all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        embs = [all_emb]
        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(norm_adj, all_emb)
            embs.append(all_emb)
        embs = torch.stack(embs, dim=1).mean(dim=1)
        return embs[:self.n_users], embs[self.n_users:]
    def bpr_loss(self, users, pos_items, neg_items, user_emb, item_emb):
        u, pos, neg = user_emb[users], item_emb[pos_items], item_emb[neg_items]
        loss = -F.logsigmoid((u * pos).sum(dim=1) - (u * neg).sum(dim=1)).mean()
        reg = 0.001 * (self.user_emb.weight[users].norm(2).pow(2) + self.item_emb.weight[pos_items].norm(2).pow(2) + self.item_emb.weight[neg_items].norm(2).pow(2)) / len(users)
        return loss + reg

def train_lightgcn(n_users, n_items, train_df, train_set, norm_adj, epochs=30, dim=64):
    coo = norm_adj.tocoo()
    norm_adj_t = torch.sparse_coo_tensor(torch.LongTensor(np.vstack([coo.row, coo.col])), torch.FloatTensor(coo.data), torch.Size(coo.shape)).to(DEVICE)
    model = LightGCN(n_users, n_items, dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LGCN_LR)
    train_users, train_items = train_df['user'].values, train_df['item'].values
    for epoch in range(epochs):
        model.train()
        user_emb, item_emb = model(norm_adj_t)
        idx = np.random.permutation(len(train_users))
        total_loss, batch_size = 0, 1024
        for i in range(0, len(idx), batch_size):
            batch_idx = idx[i:i+batch_size]
            users = torch.LongTensor(train_users[batch_idx]).to(DEVICE)
            pos_items = torch.LongTensor(train_items[batch_idx]).to(DEVICE)
            neg_items = []
            for u in train_users[batch_idx]:
                neg = np.random.randint(0, n_items)
                while (u, neg) in train_set:
                    neg = np.random.randint(0, n_items)
                neg_items.append(neg)
            neg_items = torch.LongTensor(neg_items).to(DEVICE)
            optimizer.zero_grad()
            loss = model.bpr_loss(users, pos_items, neg_items, user_emb, item_emb)
            loss.backward(retain_graph=True)
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={total_loss/(len(idx)//batch_size):.4f}")
    model.eval()
    with torch.no_grad():
        user_emb, item_emb = model(norm_adj_t)
    return model, user_emb.cpu().numpy(), item_emb.cpu().numpy()

lgcn_model, lgcn_user_emb, lgcn_item_emb = train_lightgcn(n_users, n_items, train_df, train_set, graph['norm_adj'], epochs=CFG.LGCN_EPOCHS)
print("  LightGCN ready.")

# Popularity
print("[Baseline 5] Popularity...")
item_popularity = train_df.groupby('item').size().reindex(range(n_items), fill_value=0).values
print("  Popularity ready.")

print("[Baseline 6] Content-Based...")
print("  Content-Based ready.")

training_history = {}
print("\nAll baselines ready. Proceeding to SYNAPSE-KG novel components...")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 5: SYNAPSE-KG NOVEL COMPONENTS
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("PHASE 5: SYNAPSE-KG NOVEL COMPONENTS")
print("═" * 110)

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# COMPONENT 1: DRIP (Dynamic Relational Intent Propagation)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

print("\n" + "─" * 110)
print("COMPONENT 1: DRIP (Dynamic Relational Intent Propagation)")
print("  Innovation: User intent-aware attention over interaction history")
print("─" * 110)

class DRIP(nn.Module):
    """Dynamic Relational Intent Propagation with attention mechanism."""
    def __init__(self, n_users, n_items, dim=64, n_heads=4):
        super().__init__()
        self.n_users, self.n_items, self.dim = n_users, n_items, dim

        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)

        # Attention over user history
        self.attention = nn.MultiheadAttention(dim, n_heads, dropout=0.1, batch_first=True)
        self.query_proj = nn.Linear(dim, dim)

        # Intent-aware scoring
        self.intent_mlp = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(dim, dim)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.xavier_uniform_(m.weight)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, users, items, user_histories, history_mask):
        user_e = self.user_emb(users)
        item_e = self.item_emb(items)

        # Get history embeddings
        hist_emb = self.item_emb(user_histories)  # [B, L, D]

        # Query is target item
        query = self.query_proj(item_e).unsqueeze(1)  # [B, 1, D]

        # Attention over history
        attn_out, _ = self.attention(query, hist_emb, hist_emb, key_padding_mask=~history_mask)
        attn_out = attn_out.squeeze(1)  # [B, D]

        # Combine user, item, and attended history
        combined = torch.cat([user_e + attn_out, item_e], dim=-1)
        intent_repr = self.intent_mlp(combined)

        # Final scoring
        scores = (intent_repr * item_e).sum(dim=-1)
        return scores

    def get_embeddings(self):
        return self.user_emb.weight.detach().cpu().numpy(), self.item_emb.weight.detach().cpu().numpy()

def train_drip(n_users, n_items, train_df, train_set, user_sequences, epochs=50):
    print("[DRIP] Training Dynamic Relational Intent Propagation...")

    model = DRIP(n_users, n_items, CFG.DRIP_DIM, CFG.DRIP_HEADS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.DRIP_LR, weight_decay=1e-5)

    train_users = train_df['user'].values
    train_items = train_df['item'].values
    max_hist_len = 20

    losses = []
    for epoch in range(epochs):
        model.train()
        idx = np.random.permutation(len(train_users))
        total_loss, batch_size = 0, 512

        for i in range(0, len(idx), batch_size):
            batch_idx = idx[i:i+batch_size]

            users = torch.LongTensor(train_users[batch_idx]).to(DEVICE)
            pos_items = torch.LongTensor(train_items[batch_idx]).to(DEVICE)

            # Build histories
            histories, masks = [], []
            for u in train_users[batch_idx]:
                hist = user_sequences.get(u, [0])[-max_hist_len:]
                mask = [True] * len(hist)
                hist = hist + [0] * (max_hist_len - len(hist))
                mask = mask + [False] * (max_hist_len - len(mask))
                histories.append(hist)
                masks.append(mask)
            histories = torch.LongTensor(histories).to(DEVICE)
            masks = torch.BoolTensor(masks).to(DEVICE)

            # Negative sampling
            neg_items = []
            for u in train_users[batch_idx]:
                neg = np.random.randint(0, n_items)
                while (u, neg) in train_set:
                    neg = np.random.randint(0, n_items)
                neg_items.append(neg)
            neg_items = torch.LongTensor(neg_items).to(DEVICE)

            optimizer.zero_grad()

            pos_scores = model(users, pos_items, histories, masks)
            neg_scores = model(users, neg_items, histories, masks)

            loss = -F.logsigmoid(pos_scores - neg_scores).mean()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(idx) // batch_size)
        losses.append(avg_loss)

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_loss:.4f}")

    user_emb, item_emb = model.get_embeddings()
    print(f"  DRIP embeddings: {user_emb.shape}, {item_emb.shape}")
    return model, user_emb, item_emb, losses

drip_model, drip_user_emb, drip_item_emb, drip_losses = train_drip(
    n_users, n_items, train_df, train_set, user_sequences, epochs=CFG.DRIP_EPOCHS
)
training_history['DRIP'] = drip_losses

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# COMPONENT 2: MINT (Momentum-based Interest Network Tracking)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

print("\n" + "─" * 110)
print("COMPONENT 2: MINT (Momentum-based Interest Network Tracking)")
print("  Innovation: Momentum-based contrastive learning for preference tracking")
print("─" * 110)

class MINT(nn.Module):
    """Momentum-based Interest Network Tracking with contrastive learning."""
    def __init__(self, n_users, n_items, dim=64, momentum=0.995, temperature=0.1):
        super().__init__()
        self.n_users, self.n_items, self.dim = n_users, n_items, dim
        self.momentum, self.temperature = momentum, temperature

        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)

        self.user_emb_k = nn.Embedding(n_users, dim)
        self.item_emb_k = nn.Embedding(n_items, dim)

        self.projector = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

        self._copy_params()
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def _copy_params(self):
        self.user_emb_k.weight.data.copy_(self.user_emb.weight.data)
        self.item_emb_k.weight.data.copy_(self.item_emb.weight.data)
        for param in self.user_emb_k.parameters():
            param.requires_grad = False
        for param in self.item_emb_k.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def _momentum_update(self):
        for param_q, param_k in zip(self.user_emb.parameters(), self.user_emb_k.parameters()):
            param_k.data = param_k.data * self.momentum + param_q.data * (1. - self.momentum)
        for param_q, param_k in zip(self.item_emb.parameters(), self.item_emb_k.parameters()):
            param_k.data = param_k.data * self.momentum + param_q.data * (1. - self.momentum)

    def forward(self, users, pos_items, neg_items):
        # Query embeddings
        u_q = self.user_emb(users)
        i_q = self.item_emb(pos_items)
        q = F.normalize(self.projector(u_q + i_q), dim=-1)

        # Key embeddings (momentum)
        with torch.no_grad():
            self._momentum_update()
            u_k = self.user_emb_k(users)
            i_k = self.item_emb_k(pos_items)
            k_pos = F.normalize(self.projector(u_k + i_k), dim=-1)

            # Negative keys
            u_k_neg = self.user_emb_k(users)
            i_k_neg = self.item_emb_k(neg_items)
            k_neg = F.normalize(self.projector(u_k_neg + i_k_neg), dim=-1)

        # Contrastive loss
        l_pos = (q * k_pos).sum(dim=-1, keepdim=True) / self.temperature
        l_neg = (q * k_neg).sum(dim=-1, keepdim=True) / self.temperature

        logits = torch.cat([l_pos, l_neg], dim=1)
        labels = torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device)

        return F.cross_entropy(logits, labels)

    def get_embeddings(self):
        return self.user_emb.weight.detach().cpu().numpy(), self.item_emb.weight.detach().cpu().numpy()

def train_mint(n_users, n_items, train_df, train_set, epochs=40):
    print("[MINT] Training Momentum-based Interest Network Tracking...")

    model = MINT(n_users, n_items, CFG.MINT_DIM, CFG.MINT_MOMENTUM, CFG.MINT_TEMPERATURE).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.MINT_LR)

    train_users = train_df['user'].values
    train_items = train_df['item'].values

    losses = []
    for epoch in range(epochs):
        model.train()
        idx = np.random.permutation(len(train_users))
        total_loss, batch_size = 0, 512

        for i in range(0, len(idx), batch_size):
            batch_idx = idx[i:i+batch_size]
            users = torch.LongTensor(train_users[batch_idx]).to(DEVICE)
            pos_items = torch.LongTensor(train_items[batch_idx]).to(DEVICE)

            # Hard negative mining: random negatives
            neg_items = []
            for u in train_users[batch_idx]:
                neg = np.random.randint(0, n_items)
                while (u, neg) in train_set:
                    neg = np.random.randint(0, n_items)
                neg_items.append(neg)
            neg_items = torch.LongTensor(neg_items).to(DEVICE)

            optimizer.zero_grad()
            loss = model(users, pos_items, neg_items)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(idx) // batch_size)
        losses.append(avg_loss)

        if (epoch + 1) % 8 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_loss:.4f}")

    user_emb, item_emb = model.get_embeddings()
    print(f"  MINT embeddings: {user_emb.shape}, {item_emb.shape}")
    return model, user_emb, item_emb, losses

mint_model, mint_user_emb, mint_item_emb, mint_losses = train_mint(
    n_users, n_items, train_df, train_set, epochs=CFG.MINT_EPOCHS
)
training_history['MINT'] = mint_losses

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# COMPONENT 3: BRIDGE (Bidirectional Relational Information Diffusion with Graph Encoding)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

print("\n" + "─" * 110)
print("COMPONENT 3: BRIDGE (Bidirectional Relational Information Diffusion)")
print("  Innovation: Bidirectional propagation with residual connections")
print("─" * 110)

class BRIDGE(nn.Module):
    """Bidirectional Relational Information Diffusion with Graph Encoding."""
    def __init__(self, n_users, n_items, dim=64, n_steps=3):
        super().__init__()
        self.n_users, self.n_items, self.dim, self.n_steps = n_users, n_items, dim, n_steps

        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)

        self.forward_transforms = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_steps)])
        self.backward_transforms = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_steps)])

        self.fusion = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

        self.alpha = nn.Parameter(torch.tensor(0.5))
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def propagate(self, emb, adj, transforms):
        h = emb
        outputs = [h]
        for transform in transforms:
            h_new = torch.sparse.mm(adj, h)
            h_new = transform(h_new)
            h = F.relu(h + h_new)  # Residual connection
            outputs.append(h)
        return torch.stack(outputs, dim=0).mean(dim=0)

    def forward(self, adj_norm, adj_norm_t):
        all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)

        forward_emb = self.propagate(all_emb, adj_norm, self.forward_transforms)
        backward_emb = self.propagate(all_emb, adj_norm_t, self.backward_transforms)

        alpha = torch.sigmoid(self.alpha)
        combined = torch.cat([alpha * forward_emb, (1 - alpha) * backward_emb], dim=-1)
        fused = self.fusion(combined)

        return fused[:self.n_users], fused[self.n_users:]

    def get_embeddings(self):
        return self.user_emb.weight.detach().cpu().numpy(), self.item_emb.weight.detach().cpu().numpy()

def train_bridge(n_users, n_items, train_df, train_set, norm_adj, epochs=40):
    print("[BRIDGE] Training Bidirectional Diffusion...")

    coo = norm_adj.tocoo()
    adj_norm = torch.sparse_coo_tensor(torch.LongTensor(np.vstack([coo.row, coo.col])), torch.FloatTensor(coo.data), torch.Size(coo.shape)).to(DEVICE)
    adj_norm_t = torch.sparse_coo_tensor(torch.LongTensor(np.vstack([coo.col, coo.row])), torch.FloatTensor(coo.data), torch.Size(coo.shape)).to(DEVICE)

    model = BRIDGE(n_users, n_items, CFG.BRIDGE_DIM, CFG.BRIDGE_STEPS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.BRIDGE_LR)

    train_users = train_df['user'].values
    train_items = train_df['item'].values

    losses = []
    for epoch in range(epochs):
        model.train()
        idx = np.random.permutation(len(train_users))
        total_loss, batch_size = 0, 1024

        for i in range(0, len(idx), batch_size):
            batch_idx = idx[i:i+batch_size]
            users = torch.LongTensor(train_users[batch_idx]).to(DEVICE)
            pos_items = torch.LongTensor(train_items[batch_idx]).to(DEVICE)

            neg_items = []
            for u in train_users[batch_idx]:
                neg = np.random.randint(0, n_items)
                while (u, neg) in train_set:
                    neg = np.random.randint(0, n_items)
                neg_items.append(neg)
            neg_items = torch.LongTensor(neg_items).to(DEVICE)

            optimizer.zero_grad()
            user_emb, item_emb = model(adj_norm, adj_norm_t)

            u, pos, neg = user_emb[users], item_emb[pos_items], item_emb[neg_items]
            loss = -F.logsigmoid((u * pos).sum(dim=1) - (u * neg).sum(dim=1)).mean()
            reg = 0.001 * (model.user_emb.weight[users].norm(2).pow(2) + model.item_emb.weight[pos_items].norm(2).pow(2) + model.item_emb.weight[neg_items].norm(2).pow(2)) / len(users)
            loss = loss + reg

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(idx) // batch_size)
        losses.append(avg_loss)

        if (epoch + 1) % 8 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_loss:.4f}")

    model.eval()
    with torch.no_grad():
        user_emb, item_emb = model(adj_norm, adj_norm_t)

    print(f"  BRIDGE embeddings: {user_emb.shape}, {item_emb.shape}")
    return model, user_emb.cpu().numpy(), item_emb.cpu().numpy(), losses

bridge_model, bridge_user_emb, bridge_item_emb, bridge_losses = train_bridge(
    n_users, n_items, train_df, train_set, graph['norm_adj'], epochs=CFG.BRIDGE_EPOCHS
)
training_history['BRIDGE'] = bridge_losses

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# COMPONENT 4: NEXIS (Neural EXpert Interest Synthesis)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

print("\n" + "─" * 110)
print("COMPONENT 4: NEXIS (Neural Expert Interest Synthesis)")
print("  Innovation: Multi-interest extraction with attention-based aggregation")
print("─" * 110)

class NEXIS(nn.Module):
    """Neural Expert Interest Synthesis with multi-interest extraction."""
    def __init__(self, n_users, n_items, dim=64, n_interests=4):
        super().__init__()
        self.n_users, self.n_items, self.dim, self.n_interests = n_users, n_items, dim, n_interests

        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)

        # Multi-interest extraction
        self.interest_extractors = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))
            for _ in range(n_interests)
        ])

        # Attention over interests
        self.interest_attention = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Tanh(),
            nn.Linear(dim, 1)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.xavier_uniform_(m.weight)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, users, items):
        user_e = self.user_emb(users)
        item_e = self.item_emb(items)

        # Extract multiple interests from user embedding
        interests = []
        for extractor in self.interest_extractors:
            interest = extractor(user_e)
            interests.append(interest)
        interests = torch.stack(interests, dim=1)  # [B, K, D]

        # Attention: which interest is most relevant to target item?
        item_expanded = item_e.unsqueeze(1).expand(-1, self.n_interests, -1)  # [B, K, D]
        attn_input = torch.cat([interests, item_expanded], dim=-1)  # [B, K, 2D]
        attn_scores = self.interest_attention(attn_input).squeeze(-1)  # [B, K]
        attn_weights = F.softmax(attn_scores, dim=-1)  # [B, K]

        # Weighted sum of interests
        user_final = (interests * attn_weights.unsqueeze(-1)).sum(dim=1)  # [B, D]

        scores = (user_final * item_e).sum(dim=-1)
        return scores

    def get_embeddings(self):
        return self.user_emb.weight.detach().cpu().numpy(), self.item_emb.weight.detach().cpu().numpy()

def train_nexis(n_users, n_items, train_df, train_set, epochs=40):
    print("[NEXIS] Training Neural Expert Interest Synthesis...")

    model = NEXIS(n_users, n_items, CFG.NEXIS_DIM, n_interests=4).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.NEXIS_LR)

    train_users = train_df['user'].values
    train_items = train_df['item'].values

    losses = []
    for epoch in range(epochs):
        model.train()
        idx = np.random.permutation(len(train_users))
        total_loss, batch_size = 0, 512

        for i in range(0, len(idx), batch_size):
            batch_idx = idx[i:i+batch_size]
            users = torch.LongTensor(train_users[batch_idx]).to(DEVICE)
            pos_items = torch.LongTensor(train_items[batch_idx]).to(DEVICE)

            neg_items = []
            for u in train_users[batch_idx]:
                neg = np.random.randint(0, n_items)
                while (u, neg) in train_set:
                    neg = np.random.randint(0, n_items)
                neg_items.append(neg)
            neg_items = torch.LongTensor(neg_items).to(DEVICE)

            optimizer.zero_grad()
            pos_scores = model(users, pos_items)
            neg_scores = model(users, neg_items)

            loss = -F.logsigmoid(pos_scores - neg_scores).mean()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(idx) // batch_size)
        losses.append(avg_loss)

        if (epoch + 1) % 8 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_loss:.4f}")

    user_emb, item_emb = model.get_embeddings()
    print(f"  NEXIS embeddings: {user_emb.shape}, {item_emb.shape}")
    return model, user_emb, item_emb, losses

nexis_model, nexis_user_emb, nexis_item_emb, nexis_losses = train_nexis(
    n_users, n_items, train_df, train_set, epochs=CFG.NEXIS_EPOCHS
)
training_history['NEXIS'] = nexis_losses

print("\nAll SYNAPSE-KG novel components trained successfully!")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# COMPONENT 5: ORACLE (Optimal Routing And Component-Level Ensembling)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

print("\n" + "─" * 110)
print("COMPONENT 5: ORACLE (Optimal Routing And Component-Level Ensembling)")
print("  Innovation: Deep meta-learning fusion with component-specific normalization")
print("─" * 110)

class ORACLE(nn.Module):
    """Optimal Routing And Component-Level Ensembling with deep architecture."""
    def __init__(self, n_components, hidden_dim=64, init_temp=0.5):
        super().__init__()
        self.n_components = n_components

        self.score_normalizers = nn.ModuleList([
            nn.BatchNorm1d(1) for _ in range(n_components)
        ])

        self.encoder = nn.Sequential(
            nn.Linear(n_components, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_components)
        )

        self.temperature = nn.Parameter(torch.tensor(init_temp))
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, component_scores):
        # Normalize each component score separately
        normalized = []
        for i in range(self.n_components):
            score_i = component_scores[:, i:i+1]
            normalized.append(self.score_normalizers[i](score_i))
        normalized = torch.cat(normalized, dim=1)

        h = self.encoder(normalized)

        gate_logits = self.gate(h)
        gates = F.softmax(gate_logits / self.temperature.abs().clamp(min=0.1), dim=-1)

        weighted_scores = (normalized * gates).sum(dim=-1, keepdim=True)
        direct_out = self.output(h)

        return (weighted_scores + direct_out).squeeze(-1), gates

def build_oracle_training_data(n_users, n_items, train_set, user_sequences,
                                itemknn, features, drip_user_emb, drip_item_emb,
                                mint_user_emb, mint_item_emb, bridge_user_emb, bridge_item_emb,
                                nexis_user_emb, nexis_item_emb, lgcn_user_emb, lgcn_item_emb,
                                n_pos=2000, n_neg_ratio=10):
    print("[ORACLE] Building training data...")

    rng = np.random.default_rng(CFG.SEED)

    pos_pairs = list(train_set)
    if len(pos_pairs) > n_pos:
        pos_pairs = rng.choice(pos_pairs, n_pos, replace=False).tolist()

    neg_pairs = []
    for u, i in pos_pairs:
        for _ in range(n_neg_ratio):
            neg = rng.integers(0, n_items)
            while (u, neg) in train_set:
                neg = rng.integers(0, n_items)
            neg_pairs.append((u, neg))

    all_pairs = [(u, i, 1) for u, i in pos_pairs] + [(u, i, 0) for u, i in neg_pairs]
    rng.shuffle(all_pairs)

    component_scores, labels = [], []

    for u, i, label in all_pairs:
        scores = [
            itemknn.predict(u, [i])[0],
            float(features['user_content'][u] @ features['item_content'][i]),
            features['item_popularity'][i],
            float(drip_user_emb[u] @ drip_item_emb[i]),
            float(mint_user_emb[u] @ mint_item_emb[i]),
            float(bridge_user_emb[u] @ bridge_item_emb[i]),
            float(nexis_user_emb[u] @ nexis_item_emb[i]),
            float(lgcn_user_emb[u] @ lgcn_item_emb[i])
        ]
        component_scores.append(scores)
        labels.append(label)

    print(f"  Training samples: {len(all_pairs)} ({n_pos} positive)")
    return np.array(component_scores), np.array(labels)

def train_oracle(component_scores, labels, n_components=8, epochs=100):
    print("[ORACLE] Training meta-fusion...")

    model = ORACLE(n_components, hidden_dim=CFG.ORACLE_HIDDEN, init_temp=CFG.ORACLE_TEMP).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.ORACLE_LR)
    criterion = nn.BCEWithLogitsLoss()

    # Manual normalization
    scores_mean = component_scores.mean(axis=0)
    scores_std = component_scores.std(axis=0) + 1e-8
    scores_norm = (component_scores - scores_mean) / scores_std

    X = torch.FloatTensor(scores_norm).to(DEVICE)
    y = torch.FloatTensor(labels).to(DEVICE)

    dataset_size, batch_size = len(X), 256

    losses = []
    best_loss = float('inf')
    avg_gates = None

    for epoch in range(epochs):
        model.train()
        idx = torch.randperm(dataset_size)
        total_loss = 0
        all_gates = []

        for i in range(0, dataset_size, batch_size):
            batch_idx = idx[i:i+batch_size]
            batch_X, batch_y = X[batch_idx], y[batch_idx]

            optimizer.zero_grad()
            preds, gates = model(batch_X)
            loss = criterion(preds, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            all_gates.append(gates.detach())

        avg_loss = total_loss / max(1, dataset_size // batch_size)
        losses.append(avg_loss)

        if avg_loss < best_loss:
            best_loss = avg_loss
            avg_gates = torch.cat(all_gates, dim=0).mean(dim=0).cpu().numpy()

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_loss:.4f}, Temp={model.temperature.item():.3f}")

    print(f"  ORACLE training complete. Best loss: {best_loss:.4f}")
    return model, scores_mean, scores_std, avg_gates, losses

oracle_X, oracle_y = build_oracle_training_data(
    n_users, n_items, train_set, user_sequences, itemknn, features,
    drip_user_emb, drip_item_emb, mint_user_emb, mint_item_emb,
    bridge_user_emb, bridge_item_emb, nexis_user_emb, nexis_item_emb,
    lgcn_user_emb, lgcn_item_emb
)

oracle_model, oracle_mean, oracle_std, oracle_weights, oracle_losses = train_oracle(
    oracle_X, oracle_y, n_components=8, epochs=CFG.ORACLE_EPOCHS
)
training_history['ORACLE'] = oracle_losses

component_names = ['ItemKNN', 'Content', 'Popularity', 'DRIP', 'MINT', 'BRIDGE', 'NEXIS', 'LightGCN']
print("\n  Learned Component Weights:")
for name, weight in zip(component_names, oracle_weights):
    print(f"    {name}: {weight:.4f}")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 6: SYNAPSE-KG FINAL SCORING
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("PHASE 6: SYNAPSE-KG FINAL SCORING")
print("═" * 110)

def synapse_kg_score(user, items, itemknn, features,
                     drip_user_emb, drip_item_emb,
                     mint_user_emb, mint_item_emb,
                     bridge_user_emb, bridge_item_emb,
                     nexis_user_emb, nexis_item_emb,
                     lgcn_user_emb, lgcn_item_emb,
                     oracle_model, oracle_mean, oracle_std):

    component_scores = []
    for item in items:
        scores = [
            itemknn.predict(user, [item])[0],
            float(features['user_content'][user] @ features['item_content'][item]),
            features['item_popularity'][item],
            float(drip_user_emb[user] @ drip_item_emb[item]),
            float(mint_user_emb[user] @ mint_item_emb[item]),
            float(bridge_user_emb[user] @ bridge_item_emb[item]),
            float(nexis_user_emb[user] @ nexis_item_emb[item]),
            float(lgcn_user_emb[user] @ lgcn_item_emb[item])
        ]
        component_scores.append(scores)

    component_scores = np.array(component_scores)
    scores_norm = (component_scores - oracle_mean) / oracle_std

    oracle_model.eval()
    with torch.no_grad():
        X = torch.FloatTensor(scores_norm).to(DEVICE)
        final_scores, _ = oracle_model(X)
        final_scores = final_scores.cpu().numpy()

    return final_scores

print("[SYNAPSE-KG] Final scoring function ready.")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 7: COMPREHENSIVE EVALUATION
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("PHASE 7: COMPREHENSIVE EVALUATION")
print("═" * 110)

def evaluate_model(score_fn, test_df, train_set, n_items, top_ks=[5, 10, 20]):
    metrics = {f'HR@{k}': [] for k in top_ks}
    metrics.update({f'NDCG@{k}': [] for k in top_ks})
    metrics['MRR@10'] = []

    for _, row in test_df.iterrows():
        user, true_item = int(row['user']), int(row['item'])

        neg_items = []
        while len(neg_items) < CFG.N_NEG:
            neg = np.random.randint(0, n_items)
            if (user, neg) not in train_set and neg != true_item:
                neg_items.append(neg)

        candidates = [true_item] + neg_items
        scores = score_fn(user, candidates)

        ranked_indices = np.argsort(-scores)
        rank = np.where(ranked_indices == 0)[0][0] + 1

        for k in top_ks:
            metrics[f'HR@{k}'].append(1.0 if rank <= k else 0.0)
            metrics[f'NDCG@{k}'].append(1.0 / np.log2(rank + 1) if rank <= k else 0.0)

        metrics['MRR@10'].append(1.0 / rank if rank <= 10 else 0.0)

    return {k: np.mean(v) for k, v in metrics.items()}

print("[Eval] Evaluating all models...")

results = {}

def pop_score(user, items):
    return np.array([item_popularity[i] for i in items])

def iknn_score(user, items):
    return itemknn.predict(user, items)

def uknn_score(user, items):
    return userknn.predict(user, items)

def content_score(user, items):
    return np.array([features['user_content'][user] @ features['item_content'][i] for i in items])

def mf_score(user, items):
    return np.array([mf_user_emb[user] @ mf_item_emb[i] for i in items])

def lgcn_score(user, items):
    return np.array([lgcn_user_emb[user] @ lgcn_item_emb[i] for i in items])

def drip_score(user, items):
    return np.array([drip_user_emb[user] @ drip_item_emb[i] for i in items])

def mint_score(user, items):
    return np.array([mint_user_emb[user] @ mint_item_emb[i] for i in items])

def bridge_score(user, items):
    return np.array([bridge_user_emb[user] @ bridge_item_emb[i] for i in items])

def nexis_score(user, items):
    return np.array([nexis_user_emb[user] @ nexis_item_emb[i] for i in items])

def synapse_score(user, items):
    return synapse_kg_score(user, items, itemknn, features,
                            drip_user_emb, drip_item_emb,
                            mint_user_emb, mint_item_emb,
                            bridge_user_emb, bridge_item_emb,
                            nexis_user_emb, nexis_item_emb,
                            lgcn_user_emb, lgcn_item_emb,
                            oracle_model, oracle_mean, oracle_std)

# Evaluate all models
for name, fn in [('Popularity', pop_score), ('ItemKNN', iknn_score), ('UserKNN', uknn_score),
                 ('Content', content_score), ('MF-BPR', mf_score), ('LightGCN', lgcn_score),
                 ('DRIP', drip_score), ('MINT', mint_score), ('BRIDGE', bridge_score),
                 ('NEXIS', nexis_score), ('SYNAPSE-KG', synapse_score)]:
    start = time.time()
    results[name] = evaluate_model(fn, test_df, train_set, n_items)
    print(f"  {name}: NDCG@10={results[name]['NDCG@10']:.4f}, HR@10={results[name]['HR@10']:.4f} ({time.time()-start:.1f}s)")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 8: RESULTS DISPLAY
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("RESULTS")
print("═" * 110)

sorted_models = sorted(results.keys(), key=lambda x: results[x]['NDCG@10'])

print(f"\n{'Model':<20} {'HR@5':>8} {'HR@10':>8} {'HR@20':>8} {'NDCG@10':>10} {'MRR@10':>10}")
print("─" * 75)

for model in sorted_models:
    r = results[model]
    prefix = "★" if model == 'SYNAPSE-KG' else " "
    print(f"{prefix}{model:<19} {r['HR@5']:>8.4f} {r['HR@10']:>8.4f} {r['HR@20']:>8.4f} {r['NDCG@10']:>10.4f} {r['MRR@10']:>10.4f}")

baselines = ['Popularity', 'ItemKNN', 'UserKNN', 'Content', 'MF-BPR', 'LightGCN']
best_baseline = max(baselines, key=lambda x: results[x]['NDCG@10'])
best_baseline_ndcg = results[best_baseline]['NDCG@10']
synapse_ndcg = results['SYNAPSE-KG']['NDCG@10']
improvement = (synapse_ndcg - best_baseline_ndcg) / best_baseline_ndcg * 100

print("\n" + "─" * 75)
print(f"Best Baseline: {best_baseline} (NDCG@10: {best_baseline_ndcg:.4f})")
print(f"SYNAPSE-KG: NDCG@10 = {synapse_ndcg:.4f} ({improvement:+.2f}%)")

results_df = pd.DataFrame([{'Model': m, **results[m]} for m in sorted_models])
results_df.to_csv(f"{CFG.OUTPUT_PATH}/tables/Table3_Main_Results.csv", index=False)
print("\n[Saved] Table 3: Main Results")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 9: ABLATION STUDY
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("ABLATION STUDY")
print("═" * 110)

ablation = {'SYNAPSE-KG (Full)': results['SYNAPSE-KG']}

def make_ablation_fn(exclude_idx):
    def fn(user, items):
        component_scores = []
        for item in items:
            scores = [
                itemknn.predict(user, [item])[0],
                float(features['user_content'][user] @ features['item_content'][item]),
                features['item_popularity'][item],
                float(drip_user_emb[user] @ drip_item_emb[item]) if exclude_idx != 3 else 0.0,
                float(mint_user_emb[user] @ mint_item_emb[item]) if exclude_idx != 4 else 0.0,
                float(bridge_user_emb[user] @ bridge_item_emb[item]) if exclude_idx != 5 else 0.0,
                float(nexis_user_emb[user] @ nexis_item_emb[item]) if exclude_idx != 6 else 0.0,
                float(lgcn_user_emb[user] @ lgcn_item_emb[item])
            ]
            component_scores.append(scores)
        component_scores = np.array(component_scores)
        scores_norm = (component_scores - oracle_mean) / oracle_std
        oracle_model.eval()
        with torch.no_grad():
            X = torch.FloatTensor(scores_norm).to(DEVICE)
            final_scores, _ = oracle_model(X)
        return final_scores.cpu().numpy()
    return fn

for name, idx in [('w/o DRIP', 3), ('w/o MINT', 4), ('w/o BRIDGE', 5), ('w/o NEXIS', 6)]:
    ablation[name] = evaluate_model(make_ablation_fn(idx), test_df, train_set, n_items)
    print(f"  {name}: NDCG@10={ablation[name]['NDCG@10']:.4f}")

ablation_df = pd.DataFrame([{'Variant': v, **ablation[v]} for v in ablation])
ablation_df.to_csv(f"{CFG.OUTPUT_PATH}/tables/Table5_Ablation_Study.csv", index=False)
print("[Saved] Table 5: Ablation Study")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 10: STATISTICAL SIGNIFICANCE TESTING
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("STATISTICAL SIGNIFICANCE TESTING")
print("═" * 110)

def compute_per_user_metrics(score_fn, test_df, train_set, n_items):
    ndcg_scores = []
    for _, row in test_df.iterrows():
        user, true_item = int(row['user']), int(row['item'])
        neg_items = []
        while len(neg_items) < CFG.N_NEG:
            neg = np.random.randint(0, n_items)
            if (user, neg) not in train_set and neg != true_item:
                neg_items.append(neg)
        candidates = [true_item] + neg_items
        scores = score_fn(user, candidates)
        ranked_indices = np.argsort(-scores)
        rank = np.where(ranked_indices == 0)[0][0] + 1
        ndcg_scores.append(1.0 / np.log2(rank + 1) if rank <= 10 else 0.0)
    return np.array(ndcg_scores)

print("[Stats] Computing per-user metrics...")
synapse_per_user = compute_per_user_metrics(synapse_score, test_df, train_set, n_items)
mf_per_user = compute_per_user_metrics(mf_score, test_df, train_set, n_items)
lgcn_per_user = compute_per_user_metrics(lgcn_score, test_df, train_set, n_items)
itemknn_per_user = compute_per_user_metrics(iknn_score, test_df, train_set, n_items)

stat_results = []
print(f"\n{'Comparison':<30} {'t-stat':>10} {'p-value':>12} {'Cohen d':>10} {'Sig':>5}")
print("─" * 75)

for name1, name2, scores1, scores2 in [('SYNAPSE-KG', 'MF-BPR', synapse_per_user, mf_per_user),
                                        ('SYNAPSE-KG', 'LightGCN', synapse_per_user, lgcn_per_user),
                                        ('SYNAPSE-KG', 'ItemKNN', synapse_per_user, itemknn_per_user)]:
    diff = scores1 - scores2
    t_stat, p_value = stats.ttest_rel(scores1, scores2)
    pooled_std = np.sqrt((scores1.std()**2 + scores2.std()**2) / 2)
    cohens_d = diff.mean() / pooled_std if pooled_std > 0 else 0
    sig = "✓" if p_value < 0.05 else ""
    print(f"{name1} vs {name2:<15} {t_stat:>10.3f} {p_value:>12.2e} {cohens_d:>10.3f} {sig:>5}")
    stat_results.append({'Comparison': f'{name1} vs {name2}', 't-statistic': t_stat, 'p-value': p_value, 'Cohen_d': cohens_d, 'Significant': p_value < 0.05})

stat_df = pd.DataFrame(stat_results)
stat_df.to_csv(f"{CFG.OUTPUT_PATH}/tables/Table4_Statistical_Tests.csv", index=False)
print("\n[Saved] Table 4: Statistical Significance Tests")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 11: GENERATING ALL FIGURES (12 Figures as PDFs)
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("GENERATING FIGURES (12 Figures)")
print("═" * 110)

# Color schemes
SYNAPSE_COLOR = '#27ae60'
NOVEL_COLORS = '#3498db'
BASELINE_COLOR = '#95a5a6'
HIGHLIGHT_COLOR = '#e74c3c'

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Figure 1: NDCG@10 Comparison (Horizontal Bar Chart)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(12, 7))
ndcg_vals = [results[m]['NDCG@10'] for m in sorted_models]
colors = [SYNAPSE_COLOR if m == 'SYNAPSE-KG' else NOVEL_COLORS if m in ['DRIP', 'MINT', 'BRIDGE', 'NEXIS'] else BASELINE_COLOR for m in sorted_models]
bars = ax1.barh(sorted_models, ndcg_vals, color=colors, edgecolor='black', linewidth=0.5)
ax1.axvline(best_baseline_ndcg, color=HIGHLIGHT_COLOR, linestyle='--', linewidth=2, label=f'Best Baseline ({best_baseline})')
for i, (model, val) in enumerate(zip(sorted_models, ndcg_vals)):
    ax1.text(val + 0.005, i, f'{val:.4f}', va='center', fontsize=9)
ax1.set_xlabel('NDCG@10', fontsize=12, fontweight='bold')
ax1.set_title('Figure 1: Model Performance Comparison (NDCG@10)', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')
ax1.set_xlim(0, max(ndcg_vals) * 1.15)
plt.tight_layout()
fig1.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure1_NDCG10_Comparison.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig1)
print("[Saved] Figure 1: NDCG@10 Comparison")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Figure 2: HR@K Comparison Across Different K Values
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(12, 7))
k_values = [5, 10, 20]
models_to_plot = ['Popularity', 'ItemKNN', 'UserKNN', 'MF-BPR', 'LightGCN', 'BRIDGE', 'SYNAPSE-KG']
x = np.arange(len(k_values))
width = 0.12
colors_hr = ['#bdc3c7', '#7f8c8d', '#34495e', '#9b59b6', '#3498db', '#2ecc71', '#27ae60']

for i, model in enumerate(models_to_plot):
    hr_vals = [results[model][f'HR@{k}'] for k in k_values]
    offset = (i - len(models_to_plot)/2 + 0.5) * width
    bars = ax2.bar(x + offset, hr_vals, width, label=model, color=colors_hr[i], edgecolor='black', linewidth=0.5)

ax2.set_xlabel('K (Top-K)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Hit Rate (HR@K)', fontsize=12, fontweight='bold')
ax2.set_title('Figure 2: Hit Rate Comparison Across Different K Values', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels([f'K={k}' for k in k_values])
ax2.legend(loc='upper left', ncol=2)
ax2.set_ylim(0, 1.0)
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig2.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure2_HRK_Comparison.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig2)
print("[Saved] Figure 2: HR@K Comparison")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Figure 3: Training Loss Curves for All Components
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
fig3, axes3 = plt.subplots(2, 3, figsize=(15, 10))
axes3 = axes3.flatten()

component_colors = {'DRIP': '#e74c3c', 'MINT': '#3498db', 'BRIDGE': '#2ecc71', 'NEXIS': '#9b59b6', 'ORACLE': '#f39c12'}

for idx, (comp_name, losses) in enumerate(training_history.items()):
    ax = axes3[idx]
    epochs = range(1, len(losses) + 1)
    ax.plot(epochs, losses, color=component_colors.get(comp_name, '#333'), linewidth=2, marker='o', markersize=3)
    ax.fill_between(epochs, losses, alpha=0.3, color=component_colors.get(comp_name, '#333'))
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel('Loss', fontsize=10)
    ax.set_title(f'{comp_name} Training Loss', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(1, len(losses))
    # Add final loss annotation
    ax.annotate(f'Final: {losses[-1]:.4f}', xy=(len(losses), losses[-1]),
                xytext=(len(losses)*0.7, losses[-1]*1.1), fontsize=9,
                arrowprops=dict(arrowstyle='->', color='gray'))

# Hide empty subplot
axes3[5].axis('off')
axes3[5].text(0.5, 0.5, 'SYNAPSE-KG\nTraining Complete', ha='center', va='center',
              fontsize=14, fontweight='bold', transform=axes3[5].transAxes)

fig3.suptitle('Figure 3: Component Training Loss Curves', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
fig3.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure3_Training_Loss_Curves.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig3)
print("[Saved] Figure 3: Training Loss Curves")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Figure 4: Ablation Study Results
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
fig4, ax4 = plt.subplots(figsize=(10, 6))
ablation_names = list(ablation.keys())
ablation_ndcg = [ablation[v]['NDCG@10'] for v in ablation_names]
ablation_hr = [ablation[v]['HR@10'] for v in ablation_names]

x = np.arange(len(ablation_names))
width = 0.35

bars1 = ax4.bar(x - width/2, ablation_ndcg, width, label='NDCG@10', color='#3498db', edgecolor='black')
bars2 = ax4.bar(x + width/2, ablation_hr, width, label='HR@10', color='#2ecc71', edgecolor='black')

# Add value labels
for bar, val in zip(bars1, ablation_ndcg):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.4f}',
             ha='center', va='bottom', fontsize=8, rotation=45)
for bar, val in zip(bars2, ablation_hr):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.4f}',
             ha='center', va='bottom', fontsize=8, rotation=45)

ax4.axhline(y=ablation['SYNAPSE-KG (Full)']['NDCG@10'], color='#e74c3c', linestyle='--', linewidth=2, label='Full Model NDCG@10')
ax4.set_xlabel('Model Variant', fontsize=12, fontweight='bold')
ax4.set_ylabel('Score', fontsize=12, fontweight='bold')
ax4.set_title('Figure 4: Ablation Study - Component Contribution Analysis', fontsize=14, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(ablation_names, rotation=15, ha='right')
ax4.legend(loc='lower right')
ax4.set_ylim(0, max(ablation_hr) * 1.15)
ax4.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig4.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure4_Ablation_Study.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig4)
print("[Saved] Figure 4: Ablation Study")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Figure 5: ORACLE Component Weights
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
fig5, (ax5a, ax5b) = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
weight_colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(component_names)))
bars = ax5a.bar(component_names, oracle_weights, color=weight_colors, edgecolor='black', linewidth=1)
for bar, weight in zip(bars, oracle_weights):
    ax5a.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{weight:.3f}',
              ha='center', va='bottom', fontsize=9, fontweight='bold')
ax5a.set_ylabel('Weight', fontsize=12, fontweight='bold')
ax5a.set_title('Component Weights (Bar Chart)', fontsize=12, fontweight='bold')
ax5a.tick_params(axis='x', rotation=45)
ax5a.set_ylim(0, max(oracle_weights) * 1.2)

# Pie chart
explode = [0.05 if w == max(oracle_weights) else 0 for w in oracle_weights]
ax5b.pie(oracle_weights, labels=component_names, autopct='%1.1f%%', colors=weight_colors,
         explode=explode, shadow=True, startangle=90)
ax5b.set_title('Component Weights Distribution (Pie Chart)', fontsize=12, fontweight='bold')

fig5.suptitle('Figure 5: ORACLE Learned Component Weights', fontsize=14, fontweight='bold')
plt.tight_layout()
fig5.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure5_Component_Weights.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig5)
print("[Saved] Figure 5: Component Weights")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Figure 6: User Group Analysis (by Activity Level)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
fig6, ax6 = plt.subplots(figsize=(12, 7))

# Compute user activity levels
user_activity = train_df.groupby('user').size()
activity_quartiles = pd.qcut(user_activity, q=4, labels=['Low', 'Medium-Low', 'Medium-High', 'High'])
user_groups = {label: activity_quartiles[activity_quartiles == label].index.tolist() for label in ['Low', 'Medium-Low', 'Medium-High', 'High']}

# Compute metrics per user group
group_results = {}
models_for_group = ['MF-BPR', 'LightGCN', 'SYNAPSE-KG']

for group_name, user_list in user_groups.items():
    group_test = test_df[test_df['user'].isin(user_list)]
    if len(group_test) > 0:
        group_results[group_name] = {}
        for model_name in models_for_group:
            if model_name == 'MF-BPR':
                score_fn = mf_score
            elif model_name == 'LightGCN':
                score_fn = lgcn_score
            else:
                score_fn = synapse_score

            ndcg_scores = []
            for _, row in group_test.iterrows():
                user, true_item = int(row['user']), int(row['item'])
                neg_items = []
                while len(neg_items) < min(CFG.N_NEG, 50):
                    neg = np.random.randint(0, n_items)
                    if (user, neg) not in train_set and neg != true_item:
                        neg_items.append(neg)
                candidates = [true_item] + neg_items
                scores = score_fn(user, candidates)
                ranked_indices = np.argsort(-scores)
                rank = np.where(ranked_indices == 0)[0][0] + 1
                ndcg_scores.append(1.0 / np.log2(rank + 1) if rank <= 10 else 0.0)
            group_results[group_name][model_name] = np.mean(ndcg_scores)

# Plot
x = np.arange(len(user_groups))
width = 0.25
colors_group = ['#9b59b6', '#3498db', '#27ae60']

for i, model_name in enumerate(models_for_group):
    vals = [group_results[g][model_name] for g in user_groups.keys()]
    ax6.bar(x + i*width, vals, width, label=model_name, color=colors_group[i], edgecolor='black')

ax6.set_xlabel('User Activity Group', fontsize=12, fontweight='bold')
ax6.set_ylabel('NDCG@10', fontsize=12, fontweight='bold')
ax6.set_title('Figure 6: Performance by User Activity Level', fontsize=14, fontweight='bold')
ax6.set_xticks(x + width)
ax6.set_xticklabels(list(user_groups.keys()))
ax6.legend()
ax6.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig6.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure6_User_Group_Analysis.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig6)
print("[Saved] Figure 6: User Group Analysis")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Figure 7: Item Group Analysis (by Popularity)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
fig7, ax7 = plt.subplots(figsize=(12, 7))

# Compute item popularity groups
item_pop_series = train_df.groupby('item').size()
pop_quartiles = pd.qcut(item_pop_series, q=4, labels=['Unpopular', 'Less Popular', 'Popular', 'Very Popular'], duplicates='drop')
item_groups = {label: pop_quartiles[pop_quartiles == label].index.tolist() for label in pop_quartiles.unique()}

# Compute metrics per item group
item_group_results = {}
for group_name, item_list in item_groups.items():
    group_test = test_df[test_df['item'].isin(item_list)]
    if len(group_test) > 0:
        item_group_results[group_name] = {}
        for model_name in models_for_group:
            if model_name == 'MF-BPR':
                score_fn = mf_score
            elif model_name == 'LightGCN':
                score_fn = lgcn_score
            else:
                score_fn = synapse_score

            ndcg_scores = []
            for _, row in group_test.iterrows():
                user, true_item = int(row['user']), int(row['item'])
                neg_items = []
                while len(neg_items) < min(CFG.N_NEG, 50):
                    neg = np.random.randint(0, n_items)
                    if (user, neg) not in train_set and neg != true_item:
                        neg_items.append(neg)
                candidates = [true_item] + neg_items
                scores = score_fn(user, candidates)
                ranked_indices = np.argsort(-scores)
                rank = np.where(ranked_indices == 0)[0][0] + 1
                ndcg_scores.append(1.0 / np.log2(rank + 1) if rank <= 10 else 0.0)
            item_group_results[group_name][model_name] = np.mean(ndcg_scores)

# Plot
x = np.arange(len(item_groups))
for i, model_name in enumerate(models_for_group):
    vals = [item_group_results.get(g, {}).get(model_name, 0) for g in item_groups.keys()]
    ax7.bar(x + i*width, vals, width, label=model_name, color=colors_group[i], edgecolor='black')

ax7.set_xlabel('Item Popularity Group', fontsize=12, fontweight='bold')
ax7.set_ylabel('NDCG@10', fontsize=12, fontweight='bold')
ax7.set_title('Figure 7: Performance by Item Popularity', fontsize=14, fontweight='bold')
ax7.set_xticks(x + width)
ax7.set_xticklabels(list(item_groups.keys()))
ax7.legend()
ax7.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig7.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure7_Item_Group_Analysis.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig7)
print("[Saved] Figure 7: Item Group Analysis")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Figure 8: Convergence Analysis (All Components)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
fig8, ax8 = plt.subplots(figsize=(12, 7))

for comp_name, losses in training_history.items():
    # Normalize losses for comparison
    normalized_losses = (np.array(losses) - min(losses)) / (max(losses) - min(losses) + 1e-8)
    epochs = range(1, len(losses) + 1)
    ax8.plot(epochs, normalized_losses, linewidth=2, marker='o', markersize=3,
             label=comp_name, color=component_colors.get(comp_name, '#333'))

ax8.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax8.set_ylabel('Normalized Loss', fontsize=12, fontweight='bold')
ax8.set_title('Figure 8: Convergence Analysis - All Components', fontsize=14, fontweight='bold')
ax8.legend(loc='upper right')
ax8.grid(True, alpha=0.3)
ax8.set_ylim(-0.05, 1.05)
plt.tight_layout()
fig8.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure8_Convergence_Analysis.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig8)
print("[Saved] Figure 8: Convergence Analysis")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Figure 9: Knowledge Graph Statistics Visualization
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
fig9, (ax9a, ax9b) = plt.subplots(1, 2, figsize=(14, 6))

# Relation distribution (bar chart)
relations = list(kg['rel_counts'].keys())
rel_counts = list(kg['rel_counts'].values())
rel_colors = plt.cm.Set3(np.linspace(0, 1, len(relations)))

bars = ax9a.bar(relations, rel_counts, color=rel_colors, edgecolor='black')
for bar, count in zip(bars, rel_counts):
    ax9a.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000, f'{count:,}',
              ha='center', va='bottom', fontsize=9, rotation=45)
ax9a.set_xlabel('Relation Type', fontsize=12, fontweight='bold')
ax9a.set_ylabel('Count', fontsize=12, fontweight='bold')
ax9a.set_title('Knowledge Graph Relation Distribution', fontsize=12, fontweight='bold')
ax9a.tick_params(axis='x', rotation=45)

# Entity distribution (pie chart)
entity_counts = {'Users': n_users, 'Items': n_items, 'Genres': kg['n_genres']}
ax9b.pie(list(entity_counts.values()), labels=list(entity_counts.keys()), autopct='%1.1f%%',
         colors=['#3498db', '#2ecc71', '#e74c3c'], shadow=True, startangle=90)
ax9b.set_title('Entity Type Distribution', fontsize=12, fontweight='bold')

fig9.suptitle('Figure 9: Knowledge Graph Statistics', fontsize=14, fontweight='bold')
plt.tight_layout()
fig9.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure9_KG_Statistics.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig9)
print("[Saved] Figure 9: KG Statistics")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Figure 10: Embedding Visualization (t-SNE)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
from sklearn.manifold import TSNE

fig10, axes10 = plt.subplots(1, 3, figsize=(18, 6))

# Sample items for visualization
n_sample = min(500, n_items)
sample_idx = np.random.choice(n_items, n_sample, replace=False)

# Get genre labels for coloring
item_genre_map = {}
for _, row in movies_subset.iterrows():
    item_idx = int(row['item'])
    primary_genre = row['genres'].split('|')[0]
    item_genre_map[item_idx] = primary_genre

sample_genres = [item_genre_map.get(i, 'Unknown') for i in sample_idx]
unique_genres = list(set(sample_genres))
genre_to_color = {g: plt.cm.tab20(i % 20) for i, g in enumerate(unique_genres)}
sample_colors = [genre_to_color[g] for g in sample_genres]

embeddings_to_plot = [
    ('BRIDGE', bridge_item_emb[sample_idx]),
    ('LightGCN', lgcn_item_emb[sample_idx]),
    ('MF-BPR', mf_item_emb[sample_idx])
]

for ax, (emb_name, emb) in zip(axes10, embeddings_to_plot):
    tsne = TSNE(n_components=2, random_state=CFG.SEED, perplexity=30, n_iter=1000)
    emb_2d = tsne.fit_transform(emb)
    ax.scatter(emb_2d[:, 0], emb_2d[:, 1], c=sample_colors, alpha=0.6, s=20)
    ax.set_title(f'{emb_name} Item Embeddings', fontsize=12, fontweight='bold')
    ax.set_xlabel('t-SNE Dim 1')
    ax.set_ylabel('t-SNE Dim 2')

# Add legend for top genres
top_genres = pd.Series(sample_genres).value_counts().head(8).index.tolist()
legend_elements = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=genre_to_color[g],
                              markersize=8, label=g) for g in top_genres]
axes10[2].legend(handles=legend_elements, loc='upper right', fontsize=8)

fig10.suptitle('Figure 10: Item Embedding Visualization (t-SNE)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig10.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure10_Embedding_tSNE.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig10)
print("[Saved] Figure 10: Embedding t-SNE")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Figure 11: Performance Metrics Radar Chart
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
fig11, ax11 = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

metrics_for_radar = ['HR@5', 'HR@10', 'HR@20', 'NDCG@10', 'MRR@10']
models_for_radar = ['MF-BPR', 'LightGCN', 'UserKNN', 'SYNAPSE-KG']
radar_colors = ['#9b59b6', '#3498db', '#f39c12', '#27ae60']

angles = np.linspace(0, 2 * np.pi, len(metrics_for_radar), endpoint=False).tolist()
angles += angles[:1]  # Complete the loop

for model, color in zip(models_for_radar, radar_colors):
    values = [results[model][m] for m in metrics_for_radar]
    values += values[:1]  # Complete the loop
    ax11.plot(angles, values, 'o-', linewidth=2, label=model, color=color)
    ax11.fill(angles, values, alpha=0.15, color=color)

ax11.set_xticks(angles[:-1])
ax11.set_xticklabels(metrics_for_radar, fontsize=11)
ax11.set_title('Figure 11: Performance Metrics Radar Chart', fontsize=14, fontweight='bold', y=1.08)
ax11.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
plt.tight_layout()
fig11.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure11_Radar_Chart.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig11)
print("[Saved] Figure 11: Radar Chart")

# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Figure 12: Statistical Significance Heatmap
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
fig12, ax12 = plt.subplots(figsize=(10, 8))

# Create pairwise comparison matrix
comparison_models = ['SYNAPSE-KG', 'MF-BPR', 'LightGCN', 'UserKNN', 'ItemKNN']
n_comp = len(comparison_models)
improvement_matrix = np.zeros((n_comp, n_comp))

for i, m1 in enumerate(comparison_models):
    for j, m2 in enumerate(comparison_models):
        if i != j:
            improvement_matrix[i, j] = ((results[m1]['NDCG@10'] - results[m2]['NDCG@10']) /
                                        results[m2]['NDCG@10'] * 100)

# Plot heatmap
im = ax12.imshow(improvement_matrix, cmap='RdYlGn', aspect='auto', vmin=-20, vmax=20)
ax12.set_xticks(range(n_comp))
ax12.set_yticks(range(n_comp))
ax12.set_xticklabels(comparison_models, rotation=45, ha='right')
ax12.set_yticklabels(comparison_models)

# Add text annotations
for i in range(n_comp):
    for j in range(n_comp):
        if i != j:
            text = ax12.text(j, i, f'{improvement_matrix[i, j]:.1f}%',
                           ha='center', va='center', fontsize=10,
                           color='white' if abs(improvement_matrix[i, j]) > 10 else 'black')

ax12.set_title('Figure 12: Pairwise Improvement Matrix (NDCG@10 %)', fontsize=14, fontweight='bold')
ax12.set_xlabel('Compared To', fontsize=12, fontweight='bold')
ax12.set_ylabel('Model', fontsize=12, fontweight='bold')

cbar = plt.colorbar(im, ax=ax12)
cbar.set_label('Relative Improvement (%)', fontsize=11)

plt.tight_layout()
fig12.savefig(f'{CFG.OUTPUT_PATH}/figures/Figure12_Improvement_Heatmap.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.close(fig12)
print("[Saved] Figure 12: Improvement Heatmap")

print("\n[✓] All 12 figures saved as PDFs!")

# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# PHASE 12: EXECUTIVE SUMMARY
# ══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 110)
print("SYNAPSE-KG EXECUTIVE SUMMARY")
print("═" * 110)

summary = f"""
╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║  SYNAPSE-KG: Semantic-Yielding Neural Attention with Path Synthesis and Encoding                                ║
║              for Knowledge Graph Enhanced Recommendations                                                        ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

📊 PERFORMANCE RESULTS
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  SYNAPSE-KG:
    HR@5:  {results['SYNAPSE-KG']['HR@5']:.4f} | HR@10:  {results['SYNAPSE-KG']['HR@10']:.4f} | HR@20:  {results['SYNAPSE-KG']['HR@20']:.4f}
    NDCG@10: {results['SYNAPSE-KG']['NDCG@10']:.4f} | MRR@10: {results['SYNAPSE-KG']['MRR@10']:.4f}

  Best External Baseline: {best_baseline} (NDCG@10: {best_baseline_ndcg:.4f})

  ═══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  ★ IMPROVEMENT: {improvement:+.2f}% over best baseline ★
  ═══════════════════════════════════════════════════════════════════════════════════════════════════════════════

⚙️ ORACLE COMPONENT WEIGHTS
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  {', '.join([f'{n}: {w:.3f}' for n, w in zip(component_names, oracle_weights)])}

════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
"""
print(summary)

with open(f'{CFG.OUTPUT_PATH}/tables/Summary.txt', 'w') as f:
    f.write(summary)

print("\n" + "═" * 110)
print("✓ SYNAPSE-KG pipeline completed successfully!")
print("═" * 110)

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  SYNAPSE-KG: Semantic-Yielding Neural Attention with Path Synthesis and Encoding
             for Knowledge Graph Enhanced Recommendations
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Device: cpu | PyTorch: 2.9.0+cpu
  Output: /content/drive/MyDrive/SYNAPSE/SYNAPSE-KG-Final
══════════════════════════════════════════════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────────────────────────────────────────────
PHASE 1: DATA LOADING
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
[Data] Loaded: 1,000,209 ratings, 3,883 movies, 6,040 users

──────────────────────────────────────────────────────────────────────────────────────────────────────────────
PHASE 1.5: DATA P